# Kaggle Training
# M Saqib Atif 23I-0769
# Ruhab Ahmed 23I-0559

### This is the kaggle ipynb for training models.
### The code for each of the model is same as in the python files of the training
### This is only to run on kaggle.

# Sequence of Execution
## 1. Cleaning: to remove any residue files.
## 2. Evaluation
## 3. Preprocssing
## 4. Model A Training
## 5. Model B Training
## 6. Exporting the working dir as a zip file

# Cleaning

In [ ]:
!rm -rf /kaggle/working/*

# Evaluation


In [1]:
"""
evaluate.py
──────────────────────────────────────────────────────────────────────────────
Full evaluation of Model A and Model B on the test split.

PRIMARY METRICS (NLP generation task — professor's requirement):
  BLEU   — n-gram precision between predicted answer and gold answer
  ROUGE  — recall-oriented overlap (ROUGE-1, ROUGE-2, ROUGE-L)
  METEOR — alignment-based metric covering synonyms and stemming

SECONDARY METRICS:
  Cosine Similarity Accuracy — TF-IDF cosine between article and options
  Binary classification: Accuracy, Precision, Recall, F1, Confusion Matrix
  4-way MCQ accuracy (ML model picks best option from OHE features)
  Train↔Test domain similarity

Usage:
  python src/evaluate.py
"""

import os
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)
from scipy.sparse import load_npz
import joblib

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from preprocessing import PROCESSED_DIR, BASE_DIR
from model_a_train import (
    compute_4way_accuracy,
    cosine_similarity_accuracy,
    compute_train_test_domain_similarity,
    compute_generation_metrics as ma_compute_generation_metrics,
    generate_questions_from_passage,
)

MODEL_A_DIR = os.path.join(BASE_DIR, 'models', 'model_a', 'traditional')
MODEL_B_DIR = os.path.join(BASE_DIR, 'models', 'model_b', 'traditional')
REPORTS_DIR = os.path.join(PROCESSED_DIR, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def print_section(title):
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print('═' * 60)


def print_subsection(title):
    print(f"\n  {'─' * 56}")
    print(f"    {title}")
    print(f"  {'─' * 56}")


def _ensure_nltk_resources():
    """Download required NLTK data if not already present."""
    import nltk
    resources = [
        ('tokenizers/punkt',     'punkt'),
        ('tokenizers/punkt_tab', 'punkt_tab'),
        ('corpora/wordnet',      'wordnet'),
        ('corpora/omw-1.4',      'omw-1.4'),
    ]
    for path, name in resources:
        try:
            nltk.data.find(path)
        except LookupError:
            try:
                nltk.download(name, quiet=True)
            except Exception:
                pass


def _score_generation_pair(ref_text, hyp_text, rouge_scorer_obj, smoother):
    import nltk
    from nltk.translate.bleu_score import sentence_bleu
    from nltk.translate.meteor_score import meteor_score

    ref_tokens = nltk.word_tokenize(str(ref_text).lower())
    hyp_tokens = nltk.word_tokenize(str(hyp_text).lower())
    if not ref_tokens or not hyp_tokens:
        return None

    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)
    try:
        met = meteor_score([ref_tokens], hyp_tokens)
    except Exception:
        met = 0.0
    rouge_out = rouge_scorer_obj.score(str(ref_text), str(hyp_text))

    return {
        'bleu': bleu,
        'meteor': met,
        'rouge1_f': rouge_out['rouge1'].fmeasure,
        'rouge1_p': rouge_out['rouge1'].precision,
        'rouge1_r': rouge_out['rouge1'].recall,
        'rouge2_f': rouge_out['rouge2'].fmeasure,
        'rouge2_p': rouge_out['rouge2'].precision,
        'rouge2_r': rouge_out['rouge2'].recall,
        'rougeL_f': rouge_out['rougeL'].fmeasure,
    }


# ─────────────────────────────────────────────────────────────────────────────
# PRIMARY METRICS — BLEU / ROUGE / METEOR
# ─────────────────────────────────────────────────────────────────────────────

def compute_generation_metrics(test_df, tfidf_vec=None, sample_n=300):
    """
    Generation metrics via model_a_train (BLEU / ROUGE / METEOR on generated answers).
    """
    print_section("PRIMARY METRICS — BLEU / ROUGE / METEOR")
    print("  Comparing generated answers (passage pipeline) vs. RACE references …")

    sample_df = test_df.sample(min(sample_n, len(test_df)), random_state=42)
    all_generated = []
    for _, row in sample_df.iterrows():
        article = str(row.get('article', ''))
        if len(article) < 50:
            continue
        all_generated.extend(generate_questions_from_passage(article, n_questions=2))

    if not all_generated:
        print("  WARNING: no generated samples.")
        return {}

    results = ma_compute_generation_metrics(all_generated, sample_df, sample_n=sample_n)
    if not results:
        return {}

    n = int(results.get('n_samples', 0))
    print(f"\n  Evaluated {n} samples (test split)")
    print(f"\n  ┌{'─'*40}┐")
    print(f"  │ {'METRIC':<28}  {'SCORE':>8} │")
    print(f"  ├{'─'*40}┤")
    print(f"  │ {'BLEU':<28}  {results.get('bleu', 0):>8.4f} │")
    print(f"  │ {'METEOR':<28}  {results.get('meteor', 0):>8.4f} │")
    print(f"  │ {'ROUGE-1  F1':<28}  {results.get('rouge1_f', 0):>8.4f} │")
    print(f"  │ {'ROUGE-2  F1':<28}  {results.get('rouge2_f', 0):>8.4f} │")
    print(f"  │ {'ROUGE-L  F1':<28}  {results.get('rougeL_f', 0):>8.4f} │")
    print(f"  └{'─'*40}┘")

    plot_payload = {
        'bleu': results.get('bleu', 0),
        'meteor': results.get('meteor', 0),
        'rouge1_f': results.get('rouge1_f', 0),
        'rouge2_f': results.get('rouge2_f', 0),
        'rougeL_f': results.get('rougeL_f', 0),
    }
    _save_generation_metrics_plot(plot_payload)
    return results


def compute_question_generation_metrics(test_df, sample_n=300, n_candidates=5):
    """
    Evaluate generated questions vs. gold questions using BLEU / ROUGE / METEOR.

    For each test sample:
      Candidates: generate_questions_from_passage(article)
      Reference : gold question from RACE dataset
      Selection : use the best-matching candidate by ROUGE-L F1
    """
    print_section("PRIMARY METRICS — BLEU / ROUGE / METEOR (Questions)")
    print("  Comparing generated questions vs. RACE gold questions …")

    _ensure_nltk_resources()

    from nltk.translate.bleu_score import SmoothingFunction
    from rouge_score import rouge_scorer as rs_lib

    rouge_scorer_obj = rs_lib.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
    )
    smoother = SmoothingFunction().method1

    sample_df = test_df.sample(min(sample_n, len(test_df)), random_state=42)

    bleu_scores, meteor_scores = [], []
    rouge1_f, rouge2_f, rougeL_f = [], [], []
    rouge1_p, rouge1_r = [], []
    rouge2_p, rouge2_r = [], []

    for _, row in sample_df.iterrows():
        article = str(row['article'])
        gold_q  = str(row['question'])

        candidates_rows = generate_questions_from_passage(
            article, n_questions=n_candidates
        )
        candidates = [d['question'] for d in candidates_rows]
        if not candidates:
            continue

        # Pick best candidate by ROUGE-L F1 to avoid penalizing diverse phrasing
        best = None
        best_rl = -1.0
        for cand in candidates:
            scores = _score_generation_pair(gold_q, cand, rouge_scorer_obj, smoother)
            if not scores:
                continue
            if scores['rougeL_f'] > best_rl:
                best_rl = scores['rougeL_f']
                best = scores

        if not best:
            continue

        bleu_scores.append(best['bleu'])
        meteor_scores.append(best['meteor'])
        rouge1_f.append(best['rouge1_f'])
        rouge1_p.append(best['rouge1_p'])
        rouge1_r.append(best['rouge1_r'])
        rouge2_f.append(best['rouge2_f'])
        rouge2_p.append(best['rouge2_p'])
        rouge2_r.append(best['rouge2_r'])
        rougeL_f.append(best['rougeL_f'])

    n = len(bleu_scores)
    if n == 0:
        print("  WARNING: no valid samples processed.")
        return {}

    results = {
        'q_bleu':      float(np.mean(bleu_scores)),
        'q_meteor':    float(np.mean(meteor_scores)),
        'q_rouge1_f':  float(np.mean(rouge1_f)),
        'q_rouge1_p':  float(np.mean(rouge1_p)),
        'q_rouge1_r':  float(np.mean(rouge1_r)),
        'q_rouge2_f':  float(np.mean(rouge2_f)),
        'q_rouge2_p':  float(np.mean(rouge2_p)),
        'q_rouge2_r':  float(np.mean(rouge2_r)),
        'q_rougeL_f':  float(np.mean(rougeL_f)),
        'n_samples':    n,
        'n_candidates': n_candidates,
    }

    print(f"\n  Evaluated {n} samples (test split)")
    print(f"\n  ┌{'─'*40}┐")
    print(f"  │ {'METRIC':<28}  {'SCORE':>8} │")
    print(f"  ├{'─'*40}┤")
    print(f"  │ {'BLEU':<28}  {results['q_bleu']:>8.4f} │")
    print(f"  │ {'METEOR':<28}  {results['q_meteor']:>8.4f} │")
    print(f"  │ {'ROUGE-1  F1':<28}  {results['q_rouge1_f']:>8.4f} │")
    print(f"  │   {'Precision':<26}  {results['q_rouge1_p']:>8.4f} │")
    print(f"  │   {'Recall':<26}  {results['q_rouge1_r']:>8.4f} │")
    print(f"  │ {'ROUGE-2  F1':<28}  {results['q_rouge2_f']:>8.4f} │")
    print(f"  │ {'ROUGE-L  F1':<28}  {results['q_rougeL_f']:>8.4f} │")
    print(f"  └{'─'*40}┘")

    return results


def _save_generation_metrics_plot(gen_metrics):
    """Bar chart for BLEU / ROUGE / METEOR scores."""
    labels = ['BLEU', 'METEOR', 'ROUGE-1\nF1', 'ROUGE-2\nF1', 'ROUGE-L\nF1']
    values = [
        gen_metrics.get('bleu', 0),
        gen_metrics.get('meteor', 0),
        gen_metrics.get('rouge1_f', 0),
        gen_metrics.get('rouge2_f', 0),
        gen_metrics.get('rougeL_f', 0),
    ]
    colors = ['#4f46e5', '#0891b2', '#059669', '#10b981', '#34d399']

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(labels, values, color=colors, alpha=0.88, width=0.55)
    ax.set_ylim(0, max(max(values) * 1.4, 0.1))
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('Generation Evaluation — BLEU / ROUGE / METEOR',
                 fontsize=13, fontweight='bold')
    ax.axhline(y=0, color='#334155', linewidth=0.8)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom',
                fontweight='bold', fontsize=10)
    plt.tight_layout()
    out = os.path.join(REPORTS_DIR, 'generation_metrics_plot.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"\n  Plot saved → {out}")


# ─────────────────────────────────────────────────────────────────────────────
# COSINE SIMILARITY ACCURACY
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_cosine_similarity(tfidf_vec, test_df, train_df=None, sample_n=500):
    """TF-IDF cosine similarity accuracy — argmax(cosine) predicts correct option."""
    print_section("COSINE SIMILARITY ACCURACY")

    eval_df = test_df.sample(min(sample_n, len(test_df)), random_state=42)
    cos = cosine_similarity_accuracy(tfidf_vec, eval_df)

    gap_flag = "correct > wrong ✓" if cos['sim_gap'] > 0 else "inverted ✗"
    print(f"\n  {'Cosine Similarity Accuracy':<35}: {cos['accuracy']:.4f}  ({cos['accuracy']*100:.1f}%)")
    print(f"  {'Avg similarity — correct option':<35}: {cos['avg_correct_sim']:.4f}")
    print(f"  {'Avg similarity — wrong options':<35}: {cos['avg_wrong_sim']:.4f}")
    print(f"  {'Similarity gap (correct − wrong)':<35}: {cos['sim_gap']:.4f}  [{gap_flag}]")

    if train_df is not None:
        domain_sim = compute_train_test_domain_similarity(
            train_df, test_df, tfidf_vec, sample_n=200
        )
        print(f"\n  Train↔Test domain similarity: {domain_sim:.4f}")
        cos['domain_similarity'] = domain_sim

    _save_cosine_sim_plot(tfidf_vec, eval_df)
    return cos


def _save_cosine_sim_plot(tfidf_vec, eval_df):
    from preprocessing import tfidf_cosine

    correct_sims, wrong_sims = [], []
    for _, row in eval_df.iterrows():
        article = str(row['article'])
        gold    = str(row['answer']).strip().upper()
        for opt in ['A', 'B', 'C', 'D']:
            sim = tfidf_cosine(article, str(row[opt]), tfidf_vec)
            (correct_sims if opt == gold else wrong_sims).append(sim)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle('TF-IDF Cosine Similarity: Correct vs Wrong Options',
                 fontsize=13, fontweight='bold')
    axes[0].hist(correct_sims, bins=30, alpha=0.75, color='#059669', label='Correct')
    axes[0].hist(wrong_sims,   bins=30, alpha=0.75, color='#dc2626', label='Wrong')
    axes[0].axvline(np.mean(correct_sims), color='#059669', lw=2, ls='--')
    axes[0].axvline(np.mean(wrong_sims),   color='#dc2626', lw=2, ls='--')
    axes[0].set_xlabel('Cosine Similarity'); axes[0].set_ylabel('Count')
    axes[0].legend(); axes[0].set_title('Distribution')
    means = [np.mean(correct_sims), np.mean(wrong_sims)]
    stds  = [np.std(correct_sims),  np.std(wrong_sims)]
    bars  = axes[1].bar(['Correct', 'Wrong'], means, yerr=stds,
                        color=['#059669', '#dc2626'], alpha=0.85,
                        capsize=8, error_kw={'linewidth': 2})
    axes[1].set_ylabel('Mean Cosine Similarity'); axes[1].set_title('Mean ± Std')
    for bar, m in zip(bars, means):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                     f'{m:.4f}', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, 'cosine_sim_plot.png'), dpi=120, bbox_inches='tight')
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# MODEL A  (binary + 4-way MCQ)
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model_a(ohe_vec, test_df, tfidf_vec=None):
    print_section("MODEL A — BINARY CLASSIFICATION & 4-WAY MCQ")

    X_te_path = os.path.join(PROCESSED_DIR, 'X_test_ohe.npz')
    y_te_path  = os.path.join(PROCESSED_DIR, 'y_test.npy')
    model_specs = [
        ('logistic_regression.pkl', 'Logistic Regression'),
        ('svm.pkl',                 'SVM'),
        ('naive_bayes.pkl',         'Naive Bayes'),
    ]
    results = {}

    if os.path.exists(X_te_path) and os.path.exists(y_te_path):
        print_subsection("Binary classification (is option correct?)")
        X_te = load_npz(X_te_path)
        y_te = np.load(y_te_path)
        for filename, label in model_specs:
            path = os.path.join(MODEL_A_DIR, filename)
            if not os.path.exists(path):
                continue
            model = joblib.load(path)
            preds = model.predict(X_te)
            acc = accuracy_score(y_te, preds)
            p   = precision_score(y_te, preds, average='macro', zero_division=0)
            r   = recall_score(y_te,    preds, average='macro', zero_division=0)
            f1  = f1_score(y_te,        preds, average='macro', zero_division=0)
            cm  = confusion_matrix(y_te, preds)
            print(f"    {label:<30}: Acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f1:.4f}")
            # Save confusion matrix plot
            fig, ax = plt.subplots(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=['Incorrect', 'Correct'],
                        yticklabels=['Incorrect', 'Correct'], ax=ax)
            ax.set_title(f'{label} — Confusion Matrix')
            ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
            plt.tight_layout()
            cm_path = os.path.join(REPORTS_DIR,
                                   f"{label.lower().replace(' ','_')}_cm.png")
            plt.savefig(cm_path, dpi=100); plt.close()
            key = label.lower().replace(' ', '_')
            results[key] = {'accuracy': acc, 'precision': p, 'recall': r,
                            'f1': f1, 'confusion_matrix': cm}

    print_subsection("4-way MCQ accuracy")
    for filename, label in model_specs:
        path = os.path.join(MODEL_A_DIR, filename)
        if not os.path.exists(path):
            continue
        model  = joblib.load(path)
        acc_4w = compute_4way_accuracy(model, ohe_vec, test_df)
        print(f"    {label:<30}: {acc_4w:.4f}  ({acc_4w*100:.1f}%)")
        key = label.lower().replace(' ', '_')
        if key not in results:
            results[key] = {}
        results[key]['4way_acc'] = acc_4w

    return results


# ─────────────────────────────────────────────────────────────────────────────
# MODEL B
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model_b():
    print_section("MODEL B — DISTRACTOR RANKER & HINT SCORER")
    path = os.path.join(MODEL_B_DIR, 'metrics.pkl')
    if not os.path.exists(path):
        print("  Not found. Run model_b_train.py first.")
        return {}
    metrics = joblib.load(path)
    d = metrics.get('distractor', {})
    h = metrics.get('hint', {})
    print(f"  Distractor Ranker: Acc={d.get('acc', 0):.4f}  F1={d.get('f1', 0):.4f}")
    print(f"  Hint Scorer:       Acc={h.get('acc', 0):.4f}")
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

def print_summary(gen_results, cos_results, ma_results):
    print_section("EVALUATION SUMMARY")
    print(f"\n  ┌{'─'*52}┐")
    print(f"  │ {'METRIC':<40}  {'VALUE':>8} │")
    print(f"  ├{'─'*52}┤")
    if gen_results:
        for key, lbl in [('bleu','★ BLEU (answer extraction)'),
                          ('meteor','★ METEOR'),
                          ('rouge1_f','★ ROUGE-1 F1'),
                          ('rouge2_f','★ ROUGE-2 F1'),
                          ('rougeL_f','★ ROUGE-L F1')]:
            print(f"  │ {lbl:<40}  {gen_results.get(key,0):>8.4f} │")
        for key, lbl in [('q_bleu','★ BLEU (question gen)'),
                          ('q_meteor','★ METEOR (question gen)'),
                          ('q_rouge1_f','★ ROUGE-1 F1 (question gen)'),
                          ('q_rouge2_f','★ ROUGE-2 F1 (question gen)'),
                          ('q_rougeL_f','★ ROUGE-L F1 (question gen)')]:
            if key in gen_results:
                print(f"  │ {lbl:<40}  {gen_results.get(key,0):>8.4f} │")
        print(f"  ├{'─'*52}┤")
    if cos_results:
        print(f"  │ {'Cosine Similarity Accuracy':<40}  {cos_results.get('accuracy',0):>8.4f} │")
        print(f"  │ {'Similarity Gap (correct−wrong)':<40}  {cos_results.get('sim_gap',0):>8.4f} │")
        print(f"  ├{'─'*52}┤")
    for key, lbl in [('logistic_regression', 'LR binary accuracy'),
                      ('svm',                 'SVM binary accuracy'),
                      ('naive_bayes',         'NB binary accuracy')]:
        v = ma_results.get(key, {}).get('accuracy')
        if v is not None:
            print(f"  │ {lbl:<40}  {v:>8.4f} │")
    print(f"  └{'─'*52}┘")


# ─────────────────────────────────────────────────────────────────────────────
# FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_full_evaluation():
    print_section("RACE RC PROJECT — FULL EVALUATION")

    ohe_path   = os.path.join(PROCESSED_DIR, 'ohe_vectorizer.pkl')
    tfidf_path = os.path.join(PROCESSED_DIR, 'tfidf_vectorizer.pkl')
    test_csv   = os.path.join(PROCESSED_DIR, 'test_clean.csv')
    train_csv  = os.path.join(PROCESSED_DIR, 'train_clean.csv')

    missing = [p for p in [ohe_path, tfidf_path, test_csv] if not os.path.exists(p)]
    if missing:
        print(f"\n  ERROR: Missing files:\n    " + '\n    '.join(missing))
        print("  Run train_pipeline.py first.")
        return {}

    with open(ohe_path,   'rb') as f: ohe_vec   = pickle.load(f)
    with open(tfidf_path, 'rb') as f: tfidf_vec = pickle.load(f)

    test_df  = pd.read_csv(test_csv)
    train_df = pd.read_csv(train_csv) if os.path.exists(train_csv) else None

    # ── PRIMARY: BLEU / ROUGE / METEOR ────────────────────────────────────
    gen_results = {}
    try:
        gen_results = compute_generation_metrics(test_df, tfidf_vec, sample_n=300)
        q_results = compute_question_generation_metrics(test_df, sample_n=300)
        if q_results:
            gen_results.update(q_results)
        gen_out = os.path.join(REPORTS_DIR, 'generation_metrics.pkl')
        joblib.dump(gen_results, gen_out)
        print(f"  Generation metrics saved → {gen_out}")
    except ImportError as e:
        print(f"\n  WARNING: Could not compute generation metrics ({e}).")
        print("  Install: pip install nltk rouge-score")

    # ── COSINE SIMILARITY ACCURACY ─────────────────────────────────────────
    cos_results = evaluate_cosine_similarity(
        tfidf_vec, test_df, train_df=train_df, sample_n=500
    )

    all_metrics = {
        'generation':       gen_results,
        'cosine_similarity': cos_results,
        # 'model_a':          ma_results,
        # 'model_b':          mb_results,
    }
    out_path = os.path.join(REPORTS_DIR, 'all_metrics.pkl')
    joblib.dump(all_metrics, out_path)
    print(f"\n  All metrics saved → {out_path}")
    print_section("Evaluation complete!")
    return all_metrics


if __name__ == '__main__':
    run_full_evaluation()


Evaluation module loaded successfully.


# Preprocessing


In [37]:
"""
preprocessing.py
=================
Data ingestion, normalisation, vectorisation, and artifact export
for the reading-comprehension MCQ pipeline.

Exported symbols used by downstream modules
--------------------------------------------
  ARTIFACT_DIR          : path where processed tensors/CSVs live
  normalize             : text → clean lowercase string
  guard_str             : any value → safe non-null string
  sentence_fragments    : passage → list[str] of sentences
  word_tokens           : text → list[str] content words
  dot_cosine            : (str, str) → float  bag-of-words cosine
  vec_cosine            : (str, str, vectorizer) → float TF-IDF cosine
  make_sample_string    : (article, question, choice) → str
  apply_vectorizer      : (texts, vectorizer) → sparse matrix
  prepare_all           : full pipeline entry-point
"""

import math
import os
import pickle
import re
import string
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------------
# Directory layout
# ---------------------------------------------------------------------------

SOURCE_DIR   = "/kaggle/input/datasets/muhammadsaqibatif/og-data"
ARTIFACT_DIR = "/kaggle/working/data/processed"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# alias kept for callers that reference the old name
PROCESSED_DIR = ARTIFACT_DIR

_MAX_ROWS = None          # set an integer to cap dataset size during development

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_CHOICE_KEYS   = ("A", "B", "C", "D")
_REQUIRED_COLS = ["article", "question", "A", "B", "C", "D", "answer"]
_VALID_LABELS  = set(_CHOICE_KEYS)

_SKIP_WORDS = frozenset({
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "for", "of",
    "and", "or", "but", "this", "that", "are", "was", "were", "be",
})

_SENT_BREAK = re.compile(r"(?<=[.!?])\s+")

# ---------------------------------------------------------------------------
# Text normalisation helpers
# ---------------------------------------------------------------------------

def _is_missing(val) -> bool:
    """True when val is None or a float NaN."""
    return val is None or (isinstance(val, float) and math.isnan(val))


def normalize(raw) -> str:
    """
    Produce a clean, lowercase, punctuation-free, single-spaced string.
    None / NaN inputs return an empty string.
    """
    if _is_missing(raw):
        return ""
    collapsed = str(raw).lower().replace("nan", "")
    no_punct  = collapsed.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", no_punct).strip()


def guard_str(val) -> str:
    """
    Safely convert any value to a stripped string.
    Returns '' for None, NaN, or the literal string 'nan'.
    """
    if _is_missing(val):
        return ""
    out = str(val).strip()
    return "" if out.lower() == "nan" else out


# ---------------------------------------------------------------------------
# Sentence segmentation
# ---------------------------------------------------------------------------

def sentence_fragments(passage) -> list:
    """
    Break a passage into sentence-length fragments at terminal punctuation.
    Very short fragments (≤ 5 chars) are discarded.
    """
    txt  = passage if isinstance(passage, str) else str(passage)
    raw  = _SENT_BREAK.split(txt.replace("\n", " "))
    return [chunk.strip() for chunk in raw if len(chunk.strip()) > 5]


# ---------------------------------------------------------------------------
# Tokenisation
# ---------------------------------------------------------------------------

def word_tokens(text: str) -> list:
    """
    Produce a list of content-word tokens (stopwords removed, length > 1).
    Text is normalised before splitting.
    """
    normed = normalize(text)
    return [w for w in normed.split() if w not in _SKIP_WORDS and len(w) > 1]


# ---------------------------------------------------------------------------
# Similarity utilities
# ---------------------------------------------------------------------------

def dot_cosine(text_a: str, text_b: str) -> float:
    """
    Token-frequency cosine similarity computed without any vectorizer.
    Returns a float in [0, 1].
    """
    bag_a = Counter(word_tokens(text_a))
    bag_b = Counter(word_tokens(text_b))
    if not bag_a or not bag_b:
        return 0.0
    shared  = sum(bag_a[k] * bag_b[k] for k in bag_a if k in bag_b)
    norm_a  = math.sqrt(sum(v * v for v in bag_a.values()))
    norm_b  = math.sqrt(sum(v * v for v in bag_b.values()))
    return shared / (norm_a * norm_b + 1e-9)


def vec_cosine(text_a: str, text_b: str, vectorizer) -> float:
    """
    TF-IDF cosine similarity between two strings using a fitted vectorizer.
    Returns 0.0 when either string is empty after normalisation.
    """
    a_norm = normalize(text_a)
    b_norm = normalize(text_b)
    if not a_norm or not b_norm:
        return 0.0
    pair = vectorizer.transform([a_norm, b_norm])
    return float(cosine_similarity(pair[0], pair[1])[0][0])


# aliases so model_a can import original names
cosine_similarity_feature  = dot_cosine
tfidf_cosine_similarity    = vec_cosine


# ---------------------------------------------------------------------------
# Sample construction
# ---------------------------------------------------------------------------

def make_sample_string(article: str, question: str, choice: str) -> str:
    """Concatenate article + question + choice into a single cleaned string."""
    return normalize(f"{article} {question} {choice}")


# alias
build_one_sample = make_sample_string


def apply_vectorizer(texts, vectorizer):
    """Transform a list of strings through a fitted vectorizer."""
    return vectorizer.transform(texts)


# alias
encode_texts = apply_vectorizer


# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def _sanitise(df: pd.DataFrame) -> pd.DataFrame:
    for col in _REQUIRED_COLS:
        if col in df.columns:
            df[col] = df[col].apply(guard_str)
    mask = (
        (df["article"].str.len() > 20) &
        (df["question"].str.len() > 3) &
        (df["answer"].isin(_VALID_LABELS))
    )
    return df[mask].reset_index(drop=True)


_RAW_CSV = "/kaggle/input/datasets/muhammadsaqibatif/og-data/train.csv"

def _read_raw_csv() -> pd.DataFrame:
    return pd.read_csv(_RAW_CSV, nrows=_MAX_ROWS)


def partition_data() -> dict:
    """
    Read raw CSV, sanitise, and partition into 80 / 10 / 10 splits.
    Returns dict with keys 'train', 'val', 'test'.
    """
    raw    = _read_raw_csv()
    clean  = _sanitise(raw)
    major, holdout    = train_test_split(clean,   test_size=0.20, random_state=42)
    dev,   final_test = train_test_split(holdout, test_size=0.50, random_state=42)
    return {"train": major, "val": dev, "test": final_test}


# backward-compat alias
load_data = partition_data


# ---------------------------------------------------------------------------
# Vectorizer construction
# ---------------------------------------------------------------------------

def fit_tfidf(corpus, n_features: int = 5_000) -> TfidfVectorizer:
    """Fit a TF-IDF vectorizer (unigrams + bigrams) over the given corpus."""
    vec = TfidfVectorizer(
        max_features=n_features,
        ngram_range=(1, 2),
        stop_words="english",
    )
    vec.fit(corpus)
    return vec


# alias for callers using old name
build_tfidf_vectorizer = fit_tfidf


# ---------------------------------------------------------------------------
# Dataset builder for binary answer verification
# ---------------------------------------------------------------------------

def build_model_a_dataset(df: pd.DataFrame):
    """
    Yield (text_list, label_list) for binary answer-verification training.
    Each DataFrame row produces four samples (one per option); the gold
    option receives label 1, the rest label 0.
    """
    texts, labels = [], []
    for _, row in df.iterrows():
        gold  = str(row["answer"]).strip().upper()
        base  = normalize(row["article"]) + " " + normalize(row["question"])
        for key in _CHOICE_KEYS:
            option_text = normalize(row[key])
            if not option_text:
                continue
            texts.append(base + " " + option_text)
            labels.append(1 if key == gold else 0)
    return texts, labels


# ---------------------------------------------------------------------------
# Sparse feature matrix construction + serialisation
# ---------------------------------------------------------------------------

def _export_split(name, X, y, dest_dir):
    save_npz(os.path.join(dest_dir, f"X_{name}_ohe.npz"), X)
    np.save(os.path.join(dest_dir, f"y_{name}.npy"), np.asarray(y, dtype=np.int64))
    np.save(
        os.path.join(dest_dir, f"hc_{name}.npy"),
        np.zeros((X.shape[0], 0), dtype=np.float32),
    )


def build_training_artifacts(dest_dir: str, ohe_vocab_size: int = 5_000):
    """
    Read the three cleaned CSVs from dest_dir, build OHE count matrices,
    and write: X_*_ohe.npz, y_*.npy, hc_*.npy, ohe_vectorizer.pkl,
    tfidf_vectorizer.pkl (mirrored from tfidf.pkl).
    """
    splits = {}
    for split_name in ("train", "val", "test"):
        path = os.path.join(dest_dir, f"{split_name}_clean.csv")
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Missing {path}. Run prepare_all() first."
            )
        splits[split_name] = pd.read_csv(path)

    tr_texts, tr_y = build_model_a_dataset(splits["train"])
    va_texts, va_y = build_model_a_dataset(splits["val"])
    te_texts, te_y = build_model_a_dataset(splits["test"])

    count_vec = CountVectorizer(max_features=ohe_vocab_size, ngram_range=(1, 2))
    count_vec.fit(tr_texts)

    _export_split("train", count_vec.transform(tr_texts), tr_y, dest_dir)
    _export_split("val",   count_vec.transform(va_texts), va_y, dest_dir)
    _export_split("test",  count_vec.transform(te_texts), te_y, dest_dir)

    with open(os.path.join(dest_dir, "ohe_vectorizer.pkl"), "wb") as fh:
        pickle.dump(count_vec, fh)

    # mirror tfidf pickle under the name model_b expects
    src = os.path.join(dest_dir, "tfidf.pkl")
    dst = os.path.join(dest_dir, "tfidf_vectorizer.pkl")
    if os.path.exists(src):
        with open(src, "rb") as fh:
            saved = pickle.load(fh)
        with open(dst, "wb") as fh:
            pickle.dump(saved, fh)
    elif not os.path.exists(dst):
        raise FileNotFoundError(
            f"Cannot find {src} or {dst}. Fit a TF-IDF vectorizer first."
        )


# ---------------------------------------------------------------------------
# Full preprocessing pipeline
# ---------------------------------------------------------------------------

def prepare_all(ohe_vocab_size: int = 5_000):
    """
    End-to-end preprocessing:
      1. Load and sanitise raw CSV
      2. Fit TF-IDF on training articles
      3. Write cleaned split CSVs
      4. Build and serialise OHE sparse matrices
    """
    print("[setup] reading raw data")
    partitions = partition_data()

    print("[setup] fitting TF-IDF vectorizer")
    tfidf = fit_tfidf(partitions["train"]["article"].astype(str))

    print("[setup] writing cleaned CSVs")
    for name, frame in partitions.items():
        out = os.path.join(ARTIFACT_DIR, f"{name}_clean.csv")
        frame.to_csv(out, index=False)

    tfidf_out = os.path.join(ARTIFACT_DIR, "tfidf.pkl")
    with open(tfidf_out, "wb") as fh:
        pickle.dump(tfidf, fh)

    print("[setup] building sparse OHE artifacts")
    build_training_artifacts(ARTIFACT_DIR, ohe_vocab_size=ohe_vocab_size)

    print("[setup] complete — artifacts written to", ARTIFACT_DIR)


# alias
run_preprocessing = prepare_all


if __name__ == "__main__":
    prepare_all()

[setup] reading raw data
[setup] fitting TF-IDF vectorizer
[setup] writing cleaned CSVs
[setup] building sparse OHE artifacts
[setup] complete — artifacts written to /kaggle/working/data/processed


# Writing Preprocessing.py to local directory in kaggle/working

In [4]:
%%writefile /kaggle/working/preprocessing.py

"""
pipeline_setup.py
=================
Data ingestion, normalisation, vectorisation, and artifact export
for the reading-comprehension MCQ pipeline.

Exported symbols used by downstream modules
--------------------------------------------
  ARTIFACT_DIR          : path where processed tensors/CSVs live
  normalize             : text → clean lowercase string
  guard_str             : any value → safe non-null string
  sentence_fragments    : passage → list[str] of sentences
  word_tokens           : text → list[str] content words
  dot_cosine            : (str, str) → float  bag-of-words cosine
  vec_cosine            : (str, str, vectorizer) → float TF-IDF cosine
  make_sample_string    : (article, question, choice) → str
  apply_vectorizer      : (texts, vectorizer) → sparse matrix
  prepare_all           : full pipeline entry-point
"""

import math
import os
import pickle
import re
import string
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------------
# Directory layout
# ---------------------------------------------------------------------------

SOURCE_DIR   = "/kaggle/input/datasets/muhammadsaqibatif/og-data"
ARTIFACT_DIR = "/kaggle/working/data/processed"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# alias kept for callers that reference the old name
PROCESSED_DIR = ARTIFACT_DIR

_MAX_ROWS = None          # set an integer to cap dataset size during development

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_CHOICE_KEYS   = ("A", "B", "C", "D")
_REQUIRED_COLS = ["article", "question", "A", "B", "C", "D", "answer"]
_VALID_LABELS  = set(_CHOICE_KEYS)

_SKIP_WORDS = frozenset({
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "for", "of",
    "and", "or", "but", "this", "that", "are", "was", "were", "be",
})

_SENT_BREAK = re.compile(r"(?<=[.!?])\s+")

# ---------------------------------------------------------------------------
# Text normalisation helpers
# ---------------------------------------------------------------------------

def _is_missing(val) -> bool:
    """True when val is None or a float NaN."""
    return val is None or (isinstance(val, float) and math.isnan(val))


def normalize(raw) -> str:
    """
    Produce a clean, lowercase, punctuation-free, single-spaced string.
    None / NaN inputs return an empty string.
    """
    if _is_missing(raw):
        return ""
    collapsed = str(raw).lower().replace("nan", "")
    no_punct  = collapsed.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", no_punct).strip()


def guard_str(val) -> str:
    """
    Safely convert any value to a stripped string.
    Returns '' for None, NaN, or the literal string 'nan'.
    """
    if _is_missing(val):
        return ""
    out = str(val).strip()
    return "" if out.lower() == "nan" else out


# ---------------------------------------------------------------------------
# Sentence segmentation
# ---------------------------------------------------------------------------

def sentence_fragments(passage) -> list:
    """
    Break a passage into sentence-length fragments at terminal punctuation.
    Very short fragments (≤ 5 chars) are discarded.
    """
    txt  = passage if isinstance(passage, str) else str(passage)
    raw  = _SENT_BREAK.split(txt.replace("\n", " "))
    return [chunk.strip() for chunk in raw if len(chunk.strip()) > 5]


# ---------------------------------------------------------------------------
# Tokenisation
# ---------------------------------------------------------------------------

def word_tokens(text: str) -> list:
    """
    Produce a list of content-word tokens (stopwords removed, length > 1).
    Text is normalised before splitting.
    """
    normed = normalize(text)
    return [w for w in normed.split() if w not in _SKIP_WORDS and len(w) > 1]


# ---------------------------------------------------------------------------
# Similarity utilities
# ---------------------------------------------------------------------------

def dot_cosine(text_a: str, text_b: str) -> float:
    """
    Token-frequency cosine similarity computed without any vectorizer.
    Returns a float in [0, 1].
    """
    bag_a = Counter(word_tokens(text_a))
    bag_b = Counter(word_tokens(text_b))
    if not bag_a or not bag_b:
        return 0.0
    shared  = sum(bag_a[k] * bag_b[k] for k in bag_a if k in bag_b)
    norm_a  = math.sqrt(sum(v * v for v in bag_a.values()))
    norm_b  = math.sqrt(sum(v * v for v in bag_b.values()))
    return shared / (norm_a * norm_b + 1e-9)


def vec_cosine(text_a: str, text_b: str, vectorizer) -> float:
    """
    TF-IDF cosine similarity between two strings using a fitted vectorizer.
    Returns 0.0 when either string is empty after normalisation.
    """
    a_norm = normalize(text_a)
    b_norm = normalize(text_b)
    if not a_norm or not b_norm:
        return 0.0
    pair = vectorizer.transform([a_norm, b_norm])
    return float(cosine_similarity(pair[0], pair[1])[0][0])


# aliases so model_a can import original names
cosine_similarity_feature  = dot_cosine
tfidf_cosine_similarity    = vec_cosine


# ---------------------------------------------------------------------------
# Sample construction
# ---------------------------------------------------------------------------

def make_sample_string(article: str, question: str, choice: str) -> str:
    """Concatenate article + question + choice into a single cleaned string."""
    return normalize(f"{article} {question} {choice}")


# alias
build_one_sample = make_sample_string


def apply_vectorizer(texts, vectorizer):
    """Transform a list of strings through a fitted vectorizer."""
    return vectorizer.transform(texts)


# alias
encode_texts = apply_vectorizer


# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def _sanitise(df: pd.DataFrame) -> pd.DataFrame:
    for col in _REQUIRED_COLS:
        if col in df.columns:
            df[col] = df[col].apply(guard_str)
    mask = (
        (df["article"].str.len() > 20) &
        (df["question"].str.len() > 3) &
        (df["answer"].isin(_VALID_LABELS))
    )
    return df[mask].reset_index(drop=True)


_RAW_CSV = "/kaggle/input/datasets/muhammadsaqibatif/og-data/train.csv"

def _read_raw_csv() -> pd.DataFrame:
    return pd.read_csv(_RAW_CSV, nrows=_MAX_ROWS)


def partition_data() -> dict:
    """
    Read raw CSV, sanitise, and partition into 80 / 10 / 10 splits.
    Returns dict with keys 'train', 'val', 'test'.
    """
    raw    = _read_raw_csv()
    clean  = _sanitise(raw)
    major, holdout    = train_test_split(clean,   test_size=0.20, random_state=42)
    dev,   final_test = train_test_split(holdout, test_size=0.50, random_state=42)
    return {"train": major, "val": dev, "test": final_test}


# backward-compat alias
load_data = partition_data


# ---------------------------------------------------------------------------
# Vectorizer construction
# ---------------------------------------------------------------------------

def fit_tfidf(corpus, n_features: int = 5_000) -> TfidfVectorizer:
    """Fit a TF-IDF vectorizer (unigrams + bigrams) over the given corpus."""
    vec = TfidfVectorizer(
        max_features=n_features,
        ngram_range=(1, 2),
        stop_words="english",
    )
    vec.fit(corpus)
    return vec


# alias for callers using old name
build_tfidf_vectorizer = fit_tfidf


# ---------------------------------------------------------------------------
# Dataset builder for binary answer verification
# ---------------------------------------------------------------------------

def build_model_a_dataset(df: pd.DataFrame):
    """
    Yield (text_list, label_list) for binary answer-verification training.
    Each DataFrame row produces four samples (one per option); the gold
    option receives label 1, the rest label 0.
    """
    texts, labels = [], []
    for _, row in df.iterrows():
        gold  = str(row["answer"]).strip().upper()
        base  = normalize(row["article"]) + " " + normalize(row["question"])
        for key in _CHOICE_KEYS:
            option_text = normalize(row[key])
            if not option_text:
                continue
            texts.append(base + " " + option_text)
            labels.append(1 if key == gold else 0)
    return texts, labels


# ---------------------------------------------------------------------------
# Sparse feature matrix construction + serialisation
# ---------------------------------------------------------------------------

def _export_split(name, X, y, dest_dir):
    save_npz(os.path.join(dest_dir, f"X_{name}_ohe.npz"), X)
    np.save(os.path.join(dest_dir, f"y_{name}.npy"), np.asarray(y, dtype=np.int64))
    np.save(
        os.path.join(dest_dir, f"hc_{name}.npy"),
        np.zeros((X.shape[0], 0), dtype=np.float32),
    )


def build_training_artifacts(dest_dir: str, ohe_vocab_size: int = 5_000):
    """
    Read the three cleaned CSVs from dest_dir, build OHE count matrices,
    and write: X_*_ohe.npz, y_*.npy, hc_*.npy, ohe_vectorizer.pkl,
    tfidf_vectorizer.pkl (mirrored from tfidf.pkl).
    """
    splits = {}
    for split_name in ("train", "val", "test"):
        path = os.path.join(dest_dir, f"{split_name}_clean.csv")
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Missing {path}. Run prepare_all() first."
            )
        splits[split_name] = pd.read_csv(path)

    tr_texts, tr_y = build_model_a_dataset(splits["train"])
    va_texts, va_y = build_model_a_dataset(splits["val"])
    te_texts, te_y = build_model_a_dataset(splits["test"])

    count_vec = CountVectorizer(max_features=ohe_vocab_size, ngram_range=(1, 2))
    count_vec.fit(tr_texts)

    _export_split("train", count_vec.transform(tr_texts), tr_y, dest_dir)
    _export_split("val",   count_vec.transform(va_texts), va_y, dest_dir)
    _export_split("test",  count_vec.transform(te_texts), te_y, dest_dir)

    with open(os.path.join(dest_dir, "ohe_vectorizer.pkl"), "wb") as fh:
        pickle.dump(count_vec, fh)

    # mirror tfidf pickle under the name model_b expects
    src = os.path.join(dest_dir, "tfidf.pkl")
    dst = os.path.join(dest_dir, "tfidf_vectorizer.pkl")
    if os.path.exists(src):
        with open(src, "rb") as fh:
            saved = pickle.load(fh)
        with open(dst, "wb") as fh:
            pickle.dump(saved, fh)
    elif not os.path.exists(dst):
        raise FileNotFoundError(
            f"Cannot find {src} or {dst}. Fit a TF-IDF vectorizer first."
        )


# ---------------------------------------------------------------------------
# Full preprocessing pipeline
# ---------------------------------------------------------------------------

def prepare_all(ohe_vocab_size: int = 5_000):
    """
    End-to-end preprocessing:
      1. Load and sanitise raw CSV
      2. Fit TF-IDF on training articles
      3. Write cleaned split CSVs
      4. Build and serialise OHE sparse matrices
    """
    print("[setup] reading raw data")
    partitions = partition_data()

    print("[setup] fitting TF-IDF vectorizer")
    tfidf = fit_tfidf(partitions["train"]["article"].astype(str))

    print("[setup] writing cleaned CSVs")
    for name, frame in partitions.items():
        out = os.path.join(ARTIFACT_DIR, f"{name}_clean.csv")
        frame.to_csv(out, index=False)

    tfidf_out = os.path.join(ARTIFACT_DIR, "tfidf.pkl")
    with open(tfidf_out, "wb") as fh:
        pickle.dump(tfidf, fh)

    print("[setup] building sparse OHE artifacts")
    build_training_artifacts(ARTIFACT_DIR, ohe_vocab_size=ohe_vocab_size)

    print("[setup] complete — artifacts written to", ARTIFACT_DIR)



if __name__ == "__main__":
    prepare_all()

Overwriting /kaggle/working/preprocessing.py


# **Model A**

In [1]:
import sys
sys.path.insert(0, "/kaggle/working")

try:
    import preprocessing
    print("Import OK")
except Exception as e:
    print(f"Import failed: {type(e).__name__}: {e}")

Import OK


In [31]:
"""
verifier_a.py
=============
Answer Verification + MCQ Generation  (Model A)

Classifier ensemble
-------------------
  Logistic Regression   — binary answer scorer, balanced classes, liblinear
  Calibrated SVM        — LinearSVC with Platt-scaling for probability output
  K-Means               — unsupervised cluster analysis of OHE feature space
  Label Propagation     — semi-supervised graph label spread

Ensemble strategies
-------------------
  Soft blend    — averaged class probabilities from LR + SVM
  Hard vote     — majority rule, ties broken by LR
  Stacking      — meta-LR trained on validation-set probability outputs

MCQ generation pipeline
-----------------------
  Phase 1 — candidate sentence extraction (keyword-overlap scoring)
  Phase 2 — Wh-word template instantiation
  Phase 3 — ML / heuristic question ranking
  Phase 4 — distractor assembly from article sentences

Evaluation metrics
------------------
  Binary classification : Accuracy, Precision, Recall, Macro-F1, Exact Match
  4-way MCQ accuracy    : pick best option by predicted P(correct)
  Cosine-sim accuracy   : TF-IDF retrieval baseline
  Text generation       : BLEU, ROUGE-1/2/L, METEOR
"""

from __future__ import annotations

import math
import os
import pickle
import random
import re
import sys
import warnings
from collections import Counter
from itertools import chain
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack as sparse_hstack, load_npz
from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_similarity as _sk_cos
from sklearn.semi_supervised import LabelPropagation
from sklearn.svm import LinearSVC
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Project imports
# ---------------------------------------------------------------------------

# Local development path (commented)
# _THIS_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else "/kaggle/working/data"
# ARTIFACT_DIR = os.path.join(_THIS_DIR, "..", "data", "processed")

# Kaggle paths (active)
_THIS_DIR = "/kaggle/working/src"
ARTIFACT_DIR = "/kaggle/working/data/processed"
PROCESSED_DIR = ARTIFACT_DIR

sys.path.insert(0, _THIS_DIR)
# try:
#     from preprocessing import (
#         ARTIFACT_DIR,
#         normalize,
#         word_tokens,
#         sentence_fragments,
#         dot_cosine,
#         vec_cosine,
#         build_model_a_dataset,
#         apply_vectorizer,
#         make_sample_string,
#     )
#     PROCESSED_DIR = ARTIFACT_DIR
# except ImportError:
#     # Minimal stubs so the module loads without preprocessing.py
#     ARTIFACT_DIR  = os.path.join(_THIS_DIR, "processed")
#     PROCESSED_DIR = ARTIFACT_DIR
_SW = frozenset({
    "a", "an", "the", "is", "it", "in", "of", "to", "and", "or", "for",
    "on", "with", "as", "at", "by", "be", "was", "are", "were", "this",
    "that", "from", "but", "not", "have", "has", "had", "he", "she",
    "they", "we", "you", "i", "do", "did", "will", "its", "their",
})
def normalize(raw) -> str:
    s = str(raw).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
def word_tokens(text: str) -> List[str]:
    return [t for t in normalize(text).split() if t not in _SW and len(t) > 1]
def sentence_fragments(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", str(text).strip())
    return [p.strip() for p in parts if len(p.strip()) > 10]
def dot_cosine(a: str, b: str) -> float:
    ba, bb = Counter(word_tokens(a)), Counter(word_tokens(b))
    if not ba or not bb:
        return 0.0
    dot  = sum(ba[k] * bb[k] for k in ba if k in bb)
    norm = math.sqrt(sum(v ** 2 for v in ba.values())) * \
           math.sqrt(sum(v ** 2 for v in bb.values()))
    return dot / (norm + 1e-9)
def vec_cosine(a, b, vec) -> float:
    mats = vec.transform([str(a), str(b)])
    return float(_sk_cos(mats[0], mats[1])[0][0])
def build_model_a_dataset(*_, **__):
    raise NotImplementedError("Provide preprocessing.py")
def apply_vectorizer(texts, vec):
    return vec.transform(texts)
def make_sample_string(article, question, option):
        return normalize(f"{article} {question} {option}")

# ---------------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------------

# Local paths (commented)
# _MODEL_DEST   = os.path.join(_THIS_DIR, "..", "models", "model_a", "traditional")
# _REPORTS_DEST = os.path.join(ARTIFACT_DIR, "reports")

# Kaggle paths (active)
_MODEL_DEST   = "/kaggle/working/models/model_a/traditional"
_REPORTS_DEST = "/kaggle/working/data/processed/reports"
os.makedirs(_MODEL_DEST,   exist_ok=True)
os.makedirs(_REPORTS_DEST, exist_ok=True)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_CHOICES    = ["A", "B", "C", "D"]
_RNG_SEED   = 42

# ---------------------------------------------------------------------------
# SECTION 1 — Data loading
# ---------------------------------------------------------------------------

def _fetch_arrays():
    """Load OHE feature matrices, labels, and handcrafted features."""
    d = ARTIFACT_DIR
    files = [
        "X_train_ohe.npz", "X_val_ohe.npz", "X_test_ohe.npz",
        "y_train.npy", "y_val.npy", "y_test.npy",
        "hc_train.npy", "hc_val.npy", "hc_test.npy",
    ]
    loaders = [load_npz if f.endswith(".npz") else np.load for f in files]
    results = []
    for fname, loader in tqdm(zip(files, loaders), total=len(files), desc="loading arrays", unit="file"):
        results.append(loader(os.path.join(d, fname)))
    return tuple(results)


# keep old name available
load_processed_data = _fetch_arrays


def _integrity_check(X_tr, y_tr, X_va, y_va) -> dict:
    assert X_tr.shape[0] == len(y_tr), "Train X/y mismatch"
    assert X_va.shape[0] == len(y_va), "Val X/y mismatch"
    return {
        "train_rows":      X_tr.shape[0],
        "val_rows":        X_va.shape[0],
        "train_pos_frac":  float(np.mean(y_tr)),
        "val_pos_frac":    float(np.mean(y_va)),
    }


# ---------------------------------------------------------------------------
# SECTION 2 — Internal matrix helpers
# ---------------------------------------------------------------------------

def _subsample(X, y, ceiling: int):
    if X.shape[0] <= ceiling:
        return X, y
    rng = np.random.RandomState(_RNG_SEED)
    idx = rng.choice(X.shape[0], ceiling, replace=False)
    return X[idx], y[idx]


def _shuffle_rows(X, y):
    rng  = np.random.RandomState(_RNG_SEED)
    perm = rng.permutation(X.shape[0])
    return X[perm], y[perm]


def _to_dense(X) -> np.ndarray:
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)


def _standardise(X) -> np.ndarray:
    d   = _to_dense(X)
    mu  = d.mean(0)
    sig = d.std(0)
    return (d - mu) / (sig + 1e-9)


def _minmax_scale(X) -> np.ndarray:
    d  = _to_dense(X)
    lo = d.min(0)
    hi = d.max(0)
    return (d - lo) / (hi - lo + 1e-9)


def _log_scale(X) -> np.ndarray:
    d = _to_dense(X)
    return np.log1p(np.abs(d))


def _rescale(X, strategy: str = "none"):
    if strategy == "standardise": return _standardise(X)
    if strategy == "minmax":      return _minmax_scale(X)
    if strategy == "log":         return _log_scale(X)
    return X

# kept for compatibility
preprocess_feature_matrix = _rescale


# ---------------------------------------------------------------------------
# SECTION 3 — Supervised classifiers
# ---------------------------------------------------------------------------

def fit_logistic(X_tr, y_tr, **kw) -> LogisticRegression:
    """
    Logistic Regression for binary answer verification.
    class_weight='balanced' corrects the 3:1 wrong-to-correct label ratio.
    Capped at 140 000 rows so sparse OHE matrices remain tractable.
    """
    print("[A] LR", end="", flush=True)
    X_tr, y_tr = _subsample(X_tr, y_tr, 140_000)
    X_tr = _rescale(X_tr, kw.get("rescale", "none"))
    clf = LogisticRegression(
        max_iter     = kw.get("max_iter",      300),
        C            = kw.get("C",             1.0),
        class_weight = "balanced",
        solver       = "liblinear",
        tol          = kw.get("tol",           1e-3),
        random_state = kw.get("random_state",  _RNG_SEED),
    )
    clf.fit(X_tr, y_tr)
    print(" ✓")
    return clf

# compat alias
train_logistic_regression = fit_logistic


def fit_svm(X_tr, y_tr, **kw) -> CalibratedClassifierCV:
    """
    LinearSVC wrapped in CalibratedClassifierCV (Platt scaling) to expose
    predict_proba needed for soft voting and stacking. Capped at 100 000 rows.
    """
    print("[A] SVM", end="", flush=True)
    X_tr, y_tr = _subsample(X_tr, y_tr, 100_000)
    X_tr = _rescale(X_tr, kw.get("rescale", "none"))
    base = LinearSVC(
        max_iter     = kw.get("max_iter",      1_000),
        C            = kw.get("C",             0.5),
        class_weight = "balanced",
        tol          = kw.get("tol",           1e-3),
        random_state = kw.get("random_state",  _RNG_SEED),
    )
    clf = CalibratedClassifierCV(base, cv=kw.get("cv", 2))
    clf.fit(X_tr, y_tr)
    print(" ✓")
    return clf

# compat alias
train_svm = fit_svm


# ---------------------------------------------------------------------------
# SECTION 4 — Unsupervised: K-Means
# ---------------------------------------------------------------------------

def fit_kmeans(X_tr, k: int = 4) -> KMeans:
    """
    K-Means on OHE features — finds latent answer-pattern clusters.
    Subsampled to 8 000 points for efficiency.
    """
    print("[A] K-Means", end="", flush=True)
    cap   = min(8_000, X_tr.shape[0])
    picks = np.random.choice(X_tr.shape[0], cap, replace=False)
    Xs    = X_tr[picks]
    km    = KMeans(n_clusters=k, random_state=_RNG_SEED, n_init=10, max_iter=300)
    km.fit(Xs)
    print(" ✓")
    try:
        sil = silhouette_score(Xs, km.labels_, sample_size=min(2_000, cap))
        print(f"         cohesion={sil:.4f}")
    except Exception:
        pass
    return km

# compat alias
train_kmeans = fit_kmeans


# ---------------------------------------------------------------------------
# SECTION 5 — Semi-supervised: Label Propagation
# ---------------------------------------------------------------------------

def fit_label_propagation(
    X_tr, y_tr, unlabelled_frac: float = 0.30
) -> LabelPropagation:
    """
    Simulates partially-labelled data by masking some labels to -1,
    then propagates via a KNN graph kernel. Operates on 4 500 rows.
    """
    print("[A] LabelProp", end="", flush=True)
    n    = min(4_500, X_tr.shape[0])
    idx  = np.random.choice(X_tr.shape[0], n, replace=False)
    Xd   = _to_dense(X_tr[idx])
    ys   = y_tr[idx].copy()

    mask     = np.random.rand(n) < unlabelled_frac
    y_masked = ys.copy()
    y_masked[mask] = -1

    lp = LabelPropagation(kernel="knn", n_neighbors=7, max_iter=1_000)
    lp.fit(Xd, y_masked)
    print(" ✓")
    if mask.sum() > 0:
        score = accuracy_score(ys[mask], lp.predict(Xd[mask]))
        print(f"         propagation-score={score:.4f}")
    return lp

# compat alias
train_label_propagation = fit_label_propagation


# ---------------------------------------------------------------------------
# SECTION 6 — Ensemble strategies
# ---------------------------------------------------------------------------

def _avg_proba(clf_a, clf_b, X) -> np.ndarray:
    """Average class-probability outputs of two calibrated classifiers."""
    return (clf_a.predict_proba(X) + clf_b.predict_proba(X)) / 2.0


def soft_blend_predict(clf_a, clf_b, X) -> np.ndarray:
    return np.argmax(_avg_proba(clf_a, clf_b, X), axis=1)

# compat alias
ensemble_soft_predict = soft_blend_predict


def hard_blend_predict(clf_a, clf_b, X) -> np.ndarray:
    """Majority vote; ties resolved in favour of clf_a."""
    pa = clf_a.predict(X)
    pb = clf_b.predict(X)
    return np.where(pa == pb, pa, pa)

# compat alias
ensemble_hard_predict = hard_blend_predict


def fit_meta_learner(clf_a, clf_b, X_val, y_val) -> LogisticRegression:
    """Train a Logistic meta-learner on validation-set probability outputs."""
    print("[A] meta-LR", end="", flush=True)
    meta_X = np.column_stack([
        clf_a.predict_proba(X_val)[:, 1],
        clf_b.predict_proba(X_val)[:, 1],
    ])
    meta = LogisticRegression(max_iter=500, random_state=_RNG_SEED)
    meta.fit(meta_X, y_val)
    print(" ✓")
    return meta

# compat alias
train_stacking_meta = fit_meta_learner


def stacked_predict(meta, clf_a, clf_b, X) -> np.ndarray:
    meta_X = np.column_stack([
        clf_a.predict_proba(X)[:, 1],
        clf_b.predict_proba(X)[:, 1],
    ])
    return meta.predict(meta_X)

# compat alias
stacking_predict = stacked_predict


# ---------------------------------------------------------------------------
# SECTION 7 — Cosine-similarity accuracy
# ---------------------------------------------------------------------------

def _sentence_max_cosine(article: str, option_text: str, vec) -> float:
    """Max TF-IDF cosine over all article sentences vs. option_text."""
    sents = sentence_fragments(str(article))
    if not sents:
        return vec_cosine(article, option_text, vec)
    s_vecs = vec.transform(sents)
    o_vec  = vec.transform([str(option_text)])
    return float(_sk_cos(s_vecs, o_vec).flatten().max())


def retrieval_accuracy(
    tfidf_vec,
    df: pd.DataFrame,
    n_rows: Optional[int] = None,
    ohe_vec=None,
    sentence_level: bool = True,
    alpha: float = 0.7,
) -> dict:
    """
    For each row select the option with the highest article–option similarity;
    report accuracy, average correct-option sim, average wrong-option sim, gap.
    """
    if n_rows and len(df) > n_rows:
        df = df.sample(n_rows, random_state=_RNG_SEED)

    n_hit, correct_sims, wrong_sims = 0, [], []

    for _, row in df.iterrows():
        article    = str(row["article"])
        gold       = str(row["answer"]).strip().upper()
        options    = {o: str(row[o]) for o in _CHOICES}
        per_option: Dict[str, float] = {}

        for key, txt in options.items():
            if tfidf_vec is not None:
                t_score = (
                    _sentence_max_cosine(article, txt, tfidf_vec)
                    if sentence_level
                    else vec_cosine(article, txt, tfidf_vec)
                )
            else:
                t_score = 0.0

            if ohe_vec is not None:
                o_score = (
                    _sentence_max_cosine(article, txt, ohe_vec)
                    if sentence_level
                    else vec_cosine(article, txt, ohe_vec)
                )
                per_option[key] = alpha * t_score + (1.0 - alpha) * o_score
            else:
                per_option[key] = t_score

        predicted = max(per_option, key=per_option.get)
        if predicted == gold:
            n_hit += 1
        correct_sims.append(per_option[gold])
        wrong_sims.extend(v for k, v in per_option.items() if k != gold)

    total = len(df)
    return {
        "accuracy":        n_hit / total if total else 0.0,
        "avg_correct_sim": float(np.mean(correct_sims)) if correct_sims else 0.0,
        "avg_wrong_sim":   float(np.mean(wrong_sims))   if wrong_sims   else 0.0,
        "sim_gap": float(np.mean(correct_sims) - np.mean(wrong_sims))
                   if (correct_sims and wrong_sims) else 0.0,
    }

# compat alias
cosine_similarity_accuracy = retrieval_accuracy


def sweep_retrieval_params(tfidf_vec, ohe_vec, eval_df: pd.DataFrame) -> dict:
    """Grid-search sentence_level × alpha; return best-accuracy configuration."""
    grid = [(sl, a) for sl in (True, False) for a in (0.55, 0.70, 0.85)]
    champion: dict = {"accuracy": -1.0}
    for sent_lvl, a in grid:
        result = retrieval_accuracy(
            tfidf_vec, eval_df,
            ohe_vec=ohe_vec,
            sentence_level=sent_lvl,
            alpha=a,
        )
        if result["accuracy"] > champion["accuracy"]:
            champion = {**result, "use_sentence_max": sent_lvl, "alpha": a}
    return champion

# compat alias
tune_cosine_similarity = sweep_retrieval_params


def domain_overlap(
    train_df: pd.DataFrame,
    test_df:  pd.DataFrame,
    tfidf_vec,
    n_sample: int = 200,
) -> float:
    """
    Average max-cosine similarity from each test article to the nearest
    training article — a proxy for train/test domain overlap.
    """
    tr_corpus = train_df["article"].dropna().sample(
        min(n_sample, len(train_df)), random_state=_RNG_SEED).tolist()
    te_corpus = test_df["article"].dropna().sample(
        min(n_sample, len(test_df)),  random_state=_RNG_SEED).tolist()

    tr_vecs = tfidf_vec.transform(tr_corpus)
    te_vecs = tfidf_vec.transform(te_corpus)

    block, max_sims = 50, []
    for i in tqdm(range(0, len(te_corpus), block), desc="domain overlap", unit="chunk"):
        chunk   = te_vecs[i:i + block]
        sim_mat = _sk_cos(chunk, tr_vecs)
        max_sims.extend(sim_mat.max(axis=1).tolist())

    return float(np.mean(max_sims))

# compat alias
compute_train_test_domain_similarity = domain_overlap


# ---------------------------------------------------------------------------
# SECTION 8 — 4-way MCQ accuracy
# ---------------------------------------------------------------------------

def mcq_accuracy(clf, ohe_vec, df: pd.DataFrame, n_rows: Optional[int] = None) -> float:
    """
    For each row score all four options; pick the one with highest P(correct).
    Returns fraction of rows where the predicted option matches the gold label.
    """
    if n_rows and len(df) > n_rows:
        df = df.sample(n_rows, random_state=_RNG_SEED)
    hits = total = 0
    for _, row in df.iterrows():
        encoded = [
            make_sample_string(row["article"], row["question"], str(row[o]))
            for o in _CHOICES
        ]
        X     = ohe_vec.transform(encoded)
        probs = clf.predict_proba(X)[:, 1]
        best  = _CHOICES[int(np.argmax(probs))]
        if best == str(row["answer"]).strip().upper():
            hits += 1
        total += 1
    return hits / total if total else 0.0

# compat alias
compute_4way_accuracy = mcq_accuracy


# ---------------------------------------------------------------------------
# SECTION 9 — Binary evaluation
# ---------------------------------------------------------------------------

def score_classifier(clf, X, y, tag: str = "") -> dict:
    preds = clf.predict(X)
    acc   = accuracy_score(y, preds)
    prec  = precision_score(y, preds, average="macro", zero_division=0)
    rec   = recall_score(y,    preds, average="macro", zero_division=0)
    f1    = f1_score(y,        preds, average="macro", zero_division=0)
    em    = float(np.mean([str(p) == str(t) for p, t in zip(preds, y)]))
    if tag:
        print(f"\n  {tag}")
        for name, val in [("acc", acc), ("prec", prec), ("rec", rec), ("f1", f1), ("em", em)]:
            print(f"    {name}={val:.4f}")
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "exact_match": em, "predictions": preds}

# compat alias
evaluate_binary = score_classifier


# ---------------------------------------------------------------------------
# SECTION 10 — MCQ generation
# ---------------------------------------------------------------------------

_WH_BANK: Dict[str, List[str]] = {
    "what": [
        "What does the passage say about {topic}?",
        "What is {topic} according to the passage?",
        "What role does {topic} play in the passage?",
    ],
    "how": [
        "How is {topic} described in the passage?",
        "How does {topic} relate to the main idea?",
    ],
    "why": [
        "Why is {topic} important according to the passage?",
        "Why is {topic} mentioned in the passage?",
    ],
    "where": ["Where does {topic} occur according to the passage?"],
    "when":  ["When is {topic} relevant in the context of the passage?"],
    "who":   ["Who is associated with {topic} in the passage?"],
}

_ALL_TEMPLATES: List[str] = list(chain.from_iterable(_WH_BANK.values()))

_GEN_STOP = frozenset({
    "a","an","the","is","it","in","of","to","and","or","for","on","with","as",
    "at","by","be","was","are","were","this","that","from","but","not","have",
    "has","had","he","she","they","we","you","i","do","did","will","its",
    "their","which","who","what","how","when","where","there","these","those",
    "can","could","would","should","also","been","being",
})


def _content_words(text: str) -> List[str]:
    return [t for t in word_tokens(text) if t not in _GEN_STOP and len(t) >= 4]


def _sent_score(sentence: str, answer_vocab: set) -> float:
    s_vocab = set(_content_words(sentence))
    return len(answer_vocab & s_vocab) + len(sentence.split()) / 100.0


def _drop_short_sents(sents: List[str], min_len: int = 10) -> List[str]:
    return [s for s in sents if len(s) >= min_len]


def extract_candidate_sentences(
    article: str, anchor: str, top_k: int = 5
) -> List[Tuple[str, float]]:
    """Score each article sentence by keyword overlap with anchor text."""
    raw   = sentence_fragments(article)
    sents = _drop_short_sents(raw)
    if not sents:
        return [(article[:200], 1.0)]
    a_vocab = set(_content_words(anchor))
    ranked  = [(s, _sent_score(s, a_vocab)) for s in sents]
    ranked.sort(key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


def _pick_topic_word(sentence: str, used: set) -> str:
    pool = [t for t in _content_words(sentence) if t not in used]
    if not pool:
        pool = _content_words(sentence)
    return max(pool, key=len) if pool else "this topic"


def _extract_answer_span(sentence: str, question: str, max_words: int = 8) -> str:
    words  = str(sentence).split()
    if not words:
        return sentence[:40]
    q_set  = set(normalize(question).split())
    anchor = next((i for i, w in enumerate(words) if normalize(w) in q_set), None)
    if anchor is None:
        cands = [
            (sum(1 for w in words[s:s + max_words] if w.lower() not in _GEN_STOP), s)
            for s in range(max(0, len(words) - max_words + 1))
        ]
        _, best_start = max(cands) if cands else (0, 0)
        return " ".join(words[best_start:best_start + max_words])
    start = max(0, anchor - max_words // 2)
    end   = min(len(words), start + max_words)
    start = max(0, end - max_words)
    return " ".join(words[start:end])


# -- Question ranker features --

def _ranker_features(question: str, source_sent: str, article: str) -> np.ndarray:
    q_tok = word_tokens(question)
    s_tok = word_tokens(source_sent)
    a_tok = word_tokens(article)
    q_len = len(q_tok)
    return np.array([
        len(set(q_tok) & set(s_tok)) / (len(set(q_tok)) + 1e-9),
        len(set(q_tok) & set(a_tok)) / (len(set(q_tok)) + 1e-9),
        q_len,
        float(question.split()[0].lower() in {"what","who","where","when","why","how"})
            if question else 0.0,
        float(question.strip().endswith("?")),
        float(any(t.endswith(("ed","ing","es","tion")) for t in q_tok)),
        len(set(q_tok)) / (q_len + 1e-9),
        len(_content_words(question)) / (q_len + 1e-9),
    ], dtype=np.float32)


def _heuristic_rank(pairs: List[Tuple[str, str]], article: str) -> List[Tuple[str, str]]:
    art_vocab = set(word_tokens(article))
    def score(q, _s):
        return len(set(word_tokens(q)) & art_vocab) / (len(word_tokens(q)) + 1e-9)
    return sorted(pairs, key=lambda p: score(*p), reverse=True)


# -- Distractor assembly --

def _build_distractors(article: str, correct: str, n: int = 3) -> List[str]:
    """
    Build plausible distractors in priority order:
      1. Article sentence fragments with moderate article-answer similarity
      2. Keyword-substituted answer variants
    """
    sents       = sentence_fragments(article)
    ans_vocab   = set(word_tokens(correct))
    art_content = _content_words(article)

    candidates: List[Tuple[float, str]] = []
    for sent in sents:
        phrase = _extract_answer_span(sent, correct, max_words=6)
        if set(word_tokens(phrase)) == ans_vocab or len(phrase) <= 5:
            continue
        sim = dot_cosine(phrase, correct)
        candidates.append((sim, phrase))

    candidates.sort(reverse=True)
    chosen = [ph for sim, ph in candidates if 0.08 < sim < 0.85][:n]

    swap_pool  = list(set(art_content) - ans_vocab)
    ans_words  = correct.split()
    if len(swap_pool) >= 2 and len(ans_words) >= 2:
        for _ in range(n + 1):
            variant = ans_words[:]
            variant[random.randrange(len(variant))] = random.choice(swap_pool)
            chosen.append(" ".join(variant))

    seen, unique = set(), []
    for d in chosen:
        key = normalize(d)
        if key not in seen and key != normalize(correct):
            seen.add(key)
            unique.append(d)
        if len(unique) >= n:
            break

    while len(unique) < n:
        unique.append(f"none of the above ({len(unique) + 1})")

    return unique[:n]

# compat alias
generate_distractors = _build_distractors


def compose_questions(
    article: str,
    count: int = 5,
    ranker=None,
) -> List[dict]:
    """
    Three-phase MCQ generation:
      Phase 1 — extract candidate sentences by keyword-overlap scoring
      Phase 2 — apply Wh-word templates over topic keywords
      Phase 3 — rank with ML ranker (if available) or heuristic fallback

    Each returned dict contains:
      question, answer, correct_letter, distractors, source_sentence, options
    """
    sents = sentence_fragments(article)
    if not sents:
        sents = [article[:200]]

    richest  = max(sents, key=lambda s: len(_content_words(s)))
    a_vocab  = set(_content_words(richest))

    candidates = extract_candidate_sentences(
        article, richest, top_k=max(count * 2, 10)
    )

    templates = _ALL_TEMPLATES[:]
    random.shuffle(templates)

    raw_pairs: List[Tuple[str, str]] = []
    for i, (sent, _) in enumerate(candidates):
        topic    = _pick_topic_word(sent, a_vocab)
        template = templates[i % len(templates)]
        raw_pairs.append((template.format(topic=topic), sent))

    if ranker is not None:
        feats = np.array([_ranker_features(q, s, article) for q, s in raw_pairs])
        try:
            scores = ranker.predict_proba(feats)[:, 1]
            ranked = [p for _, p in sorted(zip(scores, raw_pairs), reverse=True)]
        except Exception:
            ranked = _heuristic_rank(raw_pairs, article)
    else:
        ranked = _heuristic_rank(raw_pairs, article)

    results, seen_keys = [], set()
    for question, src_sent in ranked:
        answer  = _extract_answer_span(src_sent, question, max_words=8)
        key     = normalize(answer)[:20]
        if key in seen_keys:
            continue
        seen_keys.add(key)

        distractors = _build_distractors(article, answer, n=3)
        opts_list   = [answer] + distractors
        random.shuffle(opts_list)
        opts_dict   = dict(zip(_CHOICES, opts_list))
        correct_ltr = next(k for k, v in opts_dict.items() if v == answer)

        results.append({
            "question":        question,
            "answer":          answer,
            "correct_letter":  correct_ltr,
            "distractors":     distractors,
            "source_sentence": src_sent,
            "options":         opts_dict,
        })
        if len(results) >= count:
            break

    if not results:
        q = "What is the main idea of the passage?"
        a = _extract_answer_span(richest, q, max_words=8)
        d = _build_distractors(article, a, n=3)
        o = dict(zip(_CHOICES, [a] + d))
        results.append({
            "question": q, "answer": a, "correct_letter": "A",
            "distractors": d, "source_sentence": richest, "options": o,
        })

    return results

# compat alias
generate_questions_from_passage = compose_questions


# ---------------------------------------------------------------------------
# SECTION 11 — Generation metrics (BLEU / ROUGE / METEOR)
# ---------------------------------------------------------------------------

def _lex_tokens(text: str) -> List[str]:
    return re.findall(r"\b\w+\b", text.lower())


def _ngram_freq(tokens: List[str], n: int) -> Counter:
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def bleu(reference: str, hypothesis: str, max_n: int = 4) -> float:
    """
    Sentence-level BLEU with modified n-gram precision + brevity penalty.
    Near-zero precisions are smoothed with 1e-9 to avoid -inf log.
    """
    ref = _lex_tokens(reference)
    hyp = _lex_tokens(hypothesis)
    if not ref or not hyp:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        h_ng = _ngram_freq(hyp, n)
        r_ng = _ngram_freq(ref, n)
        if not h_ng:
            precisions.append(0.0)
            continue
        clip = sum(min(cnt, r_ng[ng]) for ng, cnt in h_ng.items())
        precisions.append(clip / sum(h_ng.values()))
    bp       = 1.0 if len(hyp) >= len(ref) else math.exp(1 - len(ref) / len(hyp))
    smoothed = [p if p > 0 else 1e-9 for p in precisions]
    return bp * math.exp(sum(math.log(p) for p in smoothed) / max_n)

# compat alias
sentence_bleu_score = bleu


def rouge(reference: str, hypothesis: str) -> dict:
    """ROUGE-1, ROUGE-2, and ROUGE-L F1 scores."""
    ref = _lex_tokens(reference)
    hyp = _lex_tokens(hypothesis)

    def _f1_ngram(r, h, n):
        rng = _ngram_freq(r, n)
        hng = _ngram_freq(h, n)
        if not rng or not hng:
            return 0.0
        shared = sum(min(rng[k], hng[k]) for k in rng if k in hng)
        prec   = shared / sum(hng.values())
        rec    = shared / sum(rng.values())
        return 2 * prec * rec / (prec + rec + 1e-9)

    def _lcs_len(a, b):
        m, n = len(a), len(b)
        dp   = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                dp[i][j] = (
                    dp[i-1][j-1] + 1
                    if a[i-1] == b[j-1]
                    else max(dp[i-1][j], dp[i][j-1])
                )
        return dp[m][n]

    r1  = _f1_ngram(ref, hyp, 1)
    r2  = _f1_ngram(ref, hyp, 2)
    lcs = _lcs_len(ref, hyp)
    if ref and hyp:
        p_l = lcs / len(hyp)
        r_l = lcs / len(ref)
        rl  = 2 * p_l * r_l / (p_l + r_l + 1e-9)
    else:
        rl  = 0.0
    return {"rouge1_f": r1, "rouge2_f": r2, "rougeL_f": rl}

# compat alias
rouge_scores = rouge


def meteor(reference: str, hypothesis: str) -> float:
    """Simplified METEOR: precision/recall harmonic mean with fragmentation penalty."""
    ref = _lex_tokens(reference)
    hyp = _lex_tokens(hypothesis)
    if not ref or not hyp:
        return 0.0
    rc  = Counter(ref)
    hc  = Counter(hyp)
    m   = sum(min(rc[t], hc[t]) for t in rc if t in hc)
    if m == 0:
        return 0.0
    prec   = m / len(hyp)
    rec    = m / len(ref)
    fmean  = (10 * prec * rec) / (9 * prec + rec + 1e-9)
    return fmean * (1 - 0.5 / max(m, 1))

# compat alias
meteor_score = meteor


def generation_metrics(
    generated: List[dict],
    df: pd.DataFrame,
    n_sample: int = 300,
) -> dict:
    """Compare generated answers to source sentences; return aggregate scores."""
    if not generated:
        return {}
    if len(df) > n_sample:
        df = df.sample(n_sample, random_state=_RNG_SEED)

    b_vals, r1_vals, r2_vals, rl_vals, m_vals = [], [], [], [], []
    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="generation metrics", unit="row")):
        sample  = generated[i % len(generated)]
        ref     = sample["source_sentence"]
        hyp     = sample["answer"]
        b_vals.append(bleu(ref, hyp))
        rg = rouge(ref, hyp)
        r1_vals.append(rg["rouge1_f"])
        r2_vals.append(rg["rouge2_f"])
        rl_vals.append(rg["rougeL_f"])
        m_vals.append(meteor(ref, hyp))

    return {
        "bleu":      float(np.mean(b_vals)),
        "rouge1_f":  float(np.mean(r1_vals)),
        "rouge2_f":  float(np.mean(r2_vals)),
        "rougeL_f":  float(np.mean(rl_vals)),
        "meteor":    float(np.mean(m_vals)),
        "n_samples": len(b_vals),
    }

# compat alias
compute_generation_metrics = generation_metrics


# ---------------------------------------------------------------------------
# SECTION 12 — Persistence
# ---------------------------------------------------------------------------

def persist_models(lr, svm, km, lp, meta, metrics: dict):
    """Write all Model-A artifacts to _MODEL_DEST."""
    artifacts = [
        ("logistic_regression.pkl", lr),
        ("svm.pkl",                 svm),
        ("kmeans.pkl",              km),
        ("label_propagation.pkl",   lp),
        ("stacking_meta.pkl",       meta),
        ("metrics.pkl",             metrics),
    ]
    for fname, obj in tqdm(
        [(f, o) for f, o in artifacts if o is not None],
        desc="saving artifacts", unit="file",
    ):
        joblib.dump(obj, os.path.join(_MODEL_DEST, fname))
    print(f"\n  artifacts → {_MODEL_DEST}")
    
    # Export metrics to CSV
    _export_metrics_to_csv(metrics)


def _export_metrics_to_csv(metrics: dict):
    """Export all metrics to CSV files in the reports directory."""
    import os
    os.makedirs(_REPORTS_DEST, exist_ok=True)
    
    # 1. Binary classification metrics (LR and SVM)
    binary_data = []
    for model_name in ["lr", "svm"]:
        if model_name in metrics and metrics[model_name]:
            row = {"model": model_name.upper()}
            row.update({k: v for k, v in metrics[model_name].items() if k != "predictions"})
            binary_data.append(row)
    
    if binary_data:
        df_binary = pd.DataFrame(binary_data)
        csv_path = os.path.join(_REPORTS_DEST, "model_a_binary_metrics.csv")
        df_binary.to_csv(csv_path, index=False)
        print(f"    → {csv_path}")
    
    # 2. Ensemble metrics
    if "ensemble" in metrics and metrics["ensemble"]:
        ensemble_data = []
        for strategy, vals in metrics["ensemble"].items():
            row = {"strategy": strategy}
            row.update(vals)
            ensemble_data.append(row)
        
        df_ensemble = pd.DataFrame(ensemble_data)
        csv_path = os.path.join(_REPORTS_DEST, "model_a_ensemble_metrics.csv")
        df_ensemble.to_csv(csv_path, index=False)
        print(f"    → {csv_path}")
    
    # 3. Cosine retrieval metrics
    if "cosine_retrieval" in metrics and metrics["cosine_retrieval"]:
        df_cosine = pd.DataFrame([metrics["cosine_retrieval"]])
        csv_path = os.path.join(_REPORTS_DEST, "model_a_cosine_retrieval_metrics.csv")
        df_cosine.to_csv(csv_path, index=False)
        print(f"    → {csv_path}")
    
    # 4. Text generation metrics
    if "text_generation" in metrics and metrics["text_generation"]:
        df_gen = pd.DataFrame([metrics["text_generation"]])
        csv_path = os.path.join(_REPORTS_DEST, "model_a_text_generation_metrics.csv")
        df_gen.to_csv(csv_path, index=False)
        print(f"    → {csv_path}")

# compat alias
save_models = persist_models


def _load_artifact(fname: str):
    path = os.path.join(_MODEL_DEST, fname)
    return joblib.load(path) if os.path.exists(path) else None


# ---------------------------------------------------------------------------
# SECTION 13 — Main training pipeline
# ---------------------------------------------------------------------------

def train_all():
    """
    End-to-end Model-A pipeline:
      1  Load OHE feature arrays
      2  Fit LR, SVM, K-Means, Label Propagation
      3  Build soft / hard / stacking ensembles
      4  Evaluate on validation: binary metrics, 4-way MCQ, cosine accuracy
      5  Generate MCQs and score BLEU / ROUGE / METEOR
      6  Final evaluation on held-out test split
      7  Persist all artifacts
    """
    BAR = "─" * 58
    print(BAR)
    print("  verifier_a  ::  training run")
    print(BAR)

    # -- 1. Load arrays --
    print("\n(1) loading feature arrays")
    X_tr, X_va, X_te, y_tr, y_va, y_te, hc_tr, hc_va, hc_te = _fetch_arrays()
    info = _integrity_check(X_tr, y_tr, X_va, y_va)
    print(f"    train={X_tr.shape[0]:,}  val={X_va.shape[0]:,}  test={X_te.shape[0]:,}")
    print(f"    features={X_tr.shape[1]:,}")
    print(f"    pos-rate — train={info['train_pos_frac']:.3f}  val={info['val_pos_frac']:.3f}")

    # -- 2. Base classifiers --
    print("\n(2) fitting base classifiers")
    classifiers = [
        ("LR",        lambda: fit_logistic(X_tr, y_tr)),
        ("SVM",       lambda: fit_svm(X_tr, y_tr)),
        ("K-Means",   lambda: fit_kmeans(X_tr, k=4)),
        ("LabelProp", lambda: fit_label_propagation(X_tr, y_tr, unlabelled_frac=0.30)),
    ]
    results = {}
    for name, fn in tqdm(classifiers, desc="base classifiers", unit="model"):
        results[name] = fn()
    lr, svm, km, lp = results["LR"], results["SVM"], results["K-Means"], results["LabelProp"]

    # -- 3. Ensembles --
    print("\n(3) building ensemble layers")
    meta = fit_meta_learner(lr, svm, X_va, y_va)

    # -- 4. Validation snapshot --
    print("\n(4) validation snapshot")
    lr_res  = score_classifier(lr,  X_va, y_va, "Logistic Regression (val)")
    svm_res = score_classifier(svm, X_va, y_va, "SVM                 (val)")

    blend_configs = [
        ("soft",    soft_blend_predict(lr, svm, X_va)),
        ("hard",    hard_blend_predict(lr, svm, X_va)),
        ("stacked", stacked_predict(meta, lr, svm, X_va)),
    ]
    blend_metrics = {}
    for label, preds in tqdm(blend_configs, desc="ensemble eval (val)", unit="strategy"):
        a = accuracy_score(y_va, preds)
        f = f1_score(y_va, preds, average="macro", zero_division=0)
        e = float(np.mean([str(p) == str(t) for p, t in zip(preds, y_va)]))
        blend_metrics[label] = {"acc": a, "f1": f, "em": e}
        print(f"\n  {label} blend (val)\n    acc={a:.4f}  f1={f:.4f}  em={e:.4f}")

    soft_a, soft_f, soft_em = blend_metrics["soft"].values()
    hard_a, hard_f, _       = blend_metrics["hard"].values()
    stk_a,  stk_f,  _       = blend_metrics["stacked"].values()

    # Local paths (commented)
    # ohe_path   = os.path.join(ARTIFACT_DIR, "ohe_vectorizer.pkl")
    # tfidf_path = os.path.join(ARTIFACT_DIR, "tfidf_vectorizer.pkl")
    # val_csv    = os.path.join(ARTIFACT_DIR, "val_clean.csv")
    # train_csv  = os.path.join(ARTIFACT_DIR, "train_clean.csv")
    # test_csv   = os.path.join(ARTIFACT_DIR, "test_clean.csv")
    
    # Kaggle paths (active)
    ohe_path   = "/kaggle/working/data/processed/ohe_vectorizer.pkl"
    tfidf_path = "/kaggle/working/data/processed/tfidf_vectorizer.pkl"
    val_csv    = "/kaggle/working/data/processed/val_clean.csv"
    train_csv  = "/kaggle/working/data/processed/train_clean.csv"
    test_csv   = "/kaggle/working/data/processed/test_clean.csv"

    cos_metrics, lr_4w, svm_4w = {}, 0.0, 0.0
    ohe_vec = tfidf_vec = None

    if all(os.path.exists(p) for p in [ohe_path, tfidf_path, val_csv]):
        with open(ohe_path,   "rb") as fh: ohe_vec   = pickle.load(fh)
        with open(tfidf_path, "rb") as fh: tfidf_vec = pickle.load(fh)

        val_df  = pd.read_csv(val_csv)
        eval_df = val_df.sample(min(1_000, len(val_df)), random_state=_RNG_SEED)

        print("\n  4-way MCQ accuracy (val):")
        for tag, mdl in tqdm([("LR", lr), ("SVM", svm)], desc="4-way MCQ", unit="model"):
            acc = mcq_accuracy(mdl, ohe_vec, eval_df)
            print(f"    {tag:4s}: {acc:.4f}")
        lr_4w  = mcq_accuracy(lr,  ohe_vec, eval_df)
        svm_4w = mcq_accuracy(svm, ohe_vec, eval_df)

        print("\n  cosine-similarity retrieval accuracy (val):")
        best = sweep_retrieval_params(tfidf_vec, ohe_vec, eval_df)
        print(f"    sent_level={best['use_sentence_max']}  alpha={best['alpha']}")
        print(f"    acc={best['accuracy']:.4f}  "
              f"avg_correct={best['avg_correct_sim']:.4f}  "
              f"avg_wrong={best['avg_wrong_sim']:.4f}  "
              f"gap={best['sim_gap']:.4f}")
        cos_metrics = dict(best)

        if os.path.exists(train_csv) and os.path.exists(test_csv):
            tr_df = pd.read_csv(train_csv)
            te_df = pd.read_csv(test_csv)
            dom   = domain_overlap(tr_df, te_df, tfidf_vec, n_sample=400)
            print(f"\n  domain overlap (train↔test): {dom:.4f}")
            cos_metrics["domain_similarity"] = dom

    # -- 5. Question generation + text metrics --
    print("\n(5) MCQ generation checks")
    gen_m: dict = {}
    if os.path.exists(val_csv):
        gen_df  = pd.read_csv(val_csv)
        sub_df  = gen_df.sample(min(400, len(gen_df)), random_state=_RNG_SEED)
        all_gen = []
        for _, row in tqdm(sub_df.iterrows(), total=len(sub_df), desc="generating MCQs", unit="article"):
            art = str(row.get("article", ""))
            if len(art) >= 50:
                all_gen.extend(compose_questions(art, count=3))
        if all_gen:
            gen_m = generation_metrics(all_gen, sub_df)
            print(f"    generated {len(all_gen)} question-answer pairs")
            for k in ("bleu", "rouge1_f", "rouge2_f", "rougeL_f", "meteor"):
                print(f"    {k.upper():<12}: {gen_m.get(k, 0):.4f}")
            report = os.path.join(_REPORTS_DEST, "generation_metrics.pkl")
            joblib.dump(gen_m, report)
            print(f"    metrics → {report}")
        else:
            print("    (no usable articles — skipping generation eval)")
    else:
        print("    (val_clean.csv not found — skipping generation eval)")

    # -- 6. Test evaluation --
    print("\n(6) held-out test split")
    test_configs = [
        ("LR  (test)",           lr,  X_te, y_te),
        ("SVM (test)",           svm, X_te, y_te),
    ]
    for label, clf, X, y in tqdm(test_configs, desc="test eval", unit="model"):
        score_classifier(clf, X, y, label)

    for label, preds in tqdm([
        ("soft",    soft_blend_predict(lr, svm, X_te)),
        ("hard",    hard_blend_predict(lr, svm, X_te)),
        ("stacked", stacked_predict(meta, lr, svm, X_te)),
    ], desc="ensemble eval (test)", unit="strategy"):
        a = accuracy_score(y_te, preds)
        f = f1_score(y_te, preds, average="macro", zero_division=0)
        e = float(np.mean([str(p) == str(t) for p, t in zip(preds, y_te)]))
        print(f"\n  {label} blend (test)\n    acc={a:.4f}  f1={f:.4f}  em={e:.4f}")
        if label == "soft":
            soft_te_a, soft_te_f, soft_te_e = a, f, e
        elif label == "hard":
            hard_te_a, hard_te_f = a, f
        elif label == "stacked":
            stk_te_a, stk_te_f = a, f

    # -- Aggregate metrics dict --
    metrics = {
        "lr":  {**lr_res,  "4way_acc": lr_4w},
        "svm": {**svm_res, "4way_acc": svm_4w},
        "ensemble": {
            "soft":    {"val_acc": soft_a, "val_f1": soft_f, "val_em": soft_em,
                        "test_acc": soft_te_a, "test_f1": soft_te_f, "test_em": soft_te_e},
            "hard":    {"val_acc": hard_a, "val_f1": hard_f,
                        "test_acc": hard_te_a, "test_f1": hard_te_f},
            "stacked": {"val_acc": stk_a,  "val_f1": stk_f,
                        "test_acc": stk_te_a, "test_f1": stk_te_f},
        },
        "cosine_retrieval":  cos_metrics,
        "text_generation":   gen_m,
    }

    persist_models(lr, svm, km, lp, meta, metrics)
    print("\n" + "=" * 65)
    print("  verifier_a training complete")
    print(BAR)
    return lr, svm, km, lp, meta, metrics


# ---------------------------------------------------------------------------
# Public inference API
# ---------------------------------------------------------------------------

def load_model_a() -> dict:
    """Load all Model-A artifacts from disk. Returns dict keyed by model role."""
    registry = {
        "lr":      "logistic_regression.pkl",
        "svm":     "svm.pkl",
        "km":      "kmeans.pkl",
        "lp":      "label_propagation.pkl",
        "meta":    "stacking_meta.pkl",
        "metrics": "metrics.pkl",
    }
    return {alias: _load_artifact(fname) for alias, fname in registry.items()}


def verify_answer(
    article: str,
    question: str,
    option: str,
    models: dict,
    ohe_vec,
) -> dict:
    """
    Predict whether `option` is the correct answer for `question` from `article`.
    Returns prediction, per-model probabilities, soft blend, and stacked output.
    """
    sample   = ohe_vec.transform([make_sample_string(article, question, option)])
    lr_prob  = models["lr"].predict_proba(sample)[0, 1]
    svm_prob = models["svm"].predict_proba(sample)[0, 1]
    blend    = float((lr_prob + svm_prob) / 2.0)

    meta_mdl  = models.get("meta")
    meta_in   = np.array([[lr_prob, svm_prob]])
    stk_prob  = meta_mdl.predict_proba(meta_in)[0, 1] if meta_mdl else blend

    return {
        "prediction":     int(stk_prob > 0.5),
        "probability":    float(stk_prob),
        "lr_proba":       float(lr_prob),
        "svm_proba":      float(svm_prob),
        "soft_ensemble":  blend,
        "stack_ensemble": float(stk_prob),
    }


def generate_mcq(article: str, n_questions: int = 5) -> List[dict]:
    """Public API: generate n_questions MCQs from a passage."""
    return compose_questions(str(article), count=n_questions)


# ---------------------------------------------------------------------------
if __name__ == "__main__":
    train_all()

──────────────────────────────────────────────────────────
  verifier_a  ::  training run
──────────────────────────────────────────────────────────

(1) loading feature arrays


loading arrays: 100%|██████████| 9/9 [00:01<00:00,  4.83file/s]


    train=281,006  val=35,118  test=35,128
    features=5,000
    pos-rate — train=0.250  val=0.250

(2) fitting base classifiers


base classifiers:   0%|          | 0/4 [00:00<?, ?model/s]

[A] LR

...


  LR  (test)
    acc=0.5188
    prec=0.5026
    rec=0.5035
    f1=0.4771
    em=0.5188

  SVM (test)
    acc=0.7500
    prec=0.3750
    rec=0.5000
    f1=0.4286
    em=0.7500


ensemble eval (test): 100%|██████████| 3/3 [00:00<00:00, 60.73strategy/s]



  soft blend (test)
    acc=0.7498  f1=0.4291  em=0.7498

  hard blend (test)
    acc=0.5188  f1=0.4771  em=0.5188

  stacked blend (test)
    acc=0.7500  f1=0.4286  em=0.7500


saving artifacts: 100%|██████████| 6/6 [00:00<00:00, 11.04file/s]


  artifacts → /kaggle/working/models/model_a/traditional
    → /kaggle/working/data/processed/reports/model_a_binary_metrics.csv
    → /kaggle/working/data/processed/reports/model_a_ensemble_metrics.csv
    → /kaggle/working/data/processed/reports/model_a_cosine_retrieval_metrics.csv
    → /kaggle/working/data/processed/reports/model_a_text_generation_metrics.csv

  verifier_a training complete
──────────────────────────────────────────────────────────


# Model B

In [33]:
"""
scorer_b.py  —  distractor & hint scorers (batched TF-IDF, logistic regression)
"""
from __future__ import annotations

import os, pickle, sys
from collections import Counter
from typing import List, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# from preprocessing import (
#     ARTIFACT_DIR,
#     sentence_fragments,
#     word_tokens,
#     normalize,
# )

# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

# Local development paths (commented)
# _THIS_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else "/kaggle/working/data"
# ARTIFACT_DIR = os.path.normpath(os.path.join(_THIS_DIR, "..", "data", "processed"))
# _SCORER_OUT = os.path.normpath(os.path.join(_THIS_DIR, "..", "models", "model_b", "traditional"))

# Kaggle paths (active)
_THIS_DIR = "/kaggle/working/src"
ARTIFACT_DIR = "/kaggle/working/data/processed"
_SCORER_OUT = "/kaggle/working/models/model_b/traditional"

sys.path.insert(0, _THIS_DIR)
os.makedirs(_SCORER_OUT, exist_ok=True)

with open(os.path.join(ARTIFACT_DIR, "tfidf_vectorizer.pkl"), "rb") as _f:
    _TFIDF = pickle.load(_f)

# ---------------------------------------------------------------------------
# Similarity helpers
# ---------------------------------------------------------------------------

def _batch_row_cosine(mat_a, mat_b) -> np.ndarray:
    """Row-wise cosine similarity between two same-shape sparse matrices."""
    dot    = np.array(mat_a.multiply(mat_b).sum(axis=1)).flatten()
    norm_a = np.sqrt(np.array(mat_a.power(2).sum(axis=1)).flatten())
    norm_b = np.sqrt(np.array(mat_b.power(2).sum(axis=1)).flatten())
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(norm_a * norm_b > 0, dot / (norm_a * norm_b), 0.0)

def _batch_transform(*lists) -> list:
    """Transform multiple string lists in one tqdm loop; returns list of sparse matrices."""
    mats, names = [], ["cand/sent", "gold/question", "passage"]
    for i, lst in enumerate(tqdm(lists, desc="  transform", unit="matrix")):
        mats.append(_TFIDF.transform(lst))
    return mats

# ---------------------------------------------------------------------------
# Keyword / candidate helpers
# ---------------------------------------------------------------------------

_OPTIONS, _NEG_PER_ITEM, _POOL_SIZE = ["A", "B", "C", "D"], 3, 15

def top_keywords(passage: str, k: int = 10) -> List[str]:
    viable = [t for t in word_tokens(passage) if len(t) >= 4]
    return [w for w, _ in Counter(viable).most_common(k)]

def _candidate_pool(passage: str) -> List[str]:
    return [w for w in top_keywords(passage, k=_POOL_SIZE) if w]

def _phrase_relevance(phrase: str, passage: str, gold: str) -> float:
    return passage.lower().count(phrase.lower()) * 0.7 + len(set(phrase.split()) & set(gold.split())) * 0.3

# ---------------------------------------------------------------------------
# Feature vectors  (pre-computed similarities passed in as scalars)
# ---------------------------------------------------------------------------

def _dist_feats(cand: str, gold: str, passage: str, sim_cg: float, sim_cp: float) -> List[float]:
    n = len(word_tokens(passage))
    return [
        sim_cg,
        sim_cp,
        len(set(cand) & set(gold)) / (len(gold) + 1e-9),
        passage.lower().count(cand.lower()) / (n + 1e-9),
        min(len(cand) / 20.0, 1.0),
        _phrase_relevance(cand, passage, gold),
    ]

def _hint_feats(sent: str, question: str, idx: int, total: int, sim: float) -> List[float]:
    q_toks, s_toks = set(word_tokens(question)), set(word_tokens(sent))
    return [len(q_toks & s_toks) / (len(q_toks) + 1e-9), idx / max(total - 1, 1), len(s_toks) / 50.0, sim]

# ---------------------------------------------------------------------------
# Dataset builders  — BATCHED
# ---------------------------------------------------------------------------

def _build_distractor_rows(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    cands, golds, passages, labels = [], [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="  distractor collect", unit="row"):
        passage   = str(row["article"])
        gold_key  = str(row["answer"])
        gold_text = str(row[gold_key])
        wrong     = [str(row[o]) for o in _OPTIONS if o != gold_key]

        for opt in wrong:
            cands.append(opt); golds.append(gold_text); passages.append(passage); labels.append(1)

        kws = [kw for kw in _candidate_pool(passage) if 3 <= len(kw) <= 15 and kw not in wrong]
        for kw in kws[:_NEG_PER_ITEM]:
            cands.append(kw); golds.append(gold_text); passages.append(passage); labels.append(0)

    mat_c, mat_g, mat_p = _batch_transform(cands, golds, passages)
    sim_cg = _batch_row_cosine(mat_c, mat_g)
    sim_cp = _batch_row_cosine(mat_c, mat_p)

    X = np.array(
        [_dist_feats(cands[i], golds[i], passages[i], sim_cg[i], sim_cp[i])
         for i in tqdm(range(len(cands)), desc="  distractor features", unit="ex")],
        dtype=np.float32,
    )
    return X, np.array(labels, dtype=np.int32)


def _build_hint_rows(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    sentences, questions, sent_indices, total_counts = [], [], [], []
    group_starts, group_sizes = [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="  hint collect", unit="row"):
        sents = sentence_fragments(str(row["article"]))
        if not sents:
            continue
        q = str(row["question"])
        start = len(sentences)
        for idx, s in enumerate(sents):
            sentences.append(s); questions.append(q)
            sent_indices.append(idx); total_counts.append(len(sents))
        group_starts.append(start); group_sizes.append(len(sents))

    mat_s, mat_q = _batch_transform(sentences, questions)
    sims = _batch_row_cosine(mat_s, mat_q)

    labels = np.zeros(len(sentences), dtype=np.int32)
    for start, size in zip(group_starts, group_sizes):
        labels[start + int(np.argmax(sims[start:start + size]))] = 1

    X = np.array(
        [_hint_feats(sentences[i], questions[i], sent_indices[i], total_counts[i], float(sims[i]))
         for i in tqdm(range(len(sentences)), desc="  hint features", unit="ex")],
        dtype=np.float32,
    )
    return X, labels

# ---------------------------------------------------------------------------
# Training & evaluation
# ---------------------------------------------------------------------------

def _fit(X: np.ndarray, y: np.ndarray) -> LogisticRegression:
    clf = LogisticRegression(max_iter=1_500, C=2.0, class_weight="balanced")
    return clf.fit(X, y)

fit_distractor_scorer = fit_hint_scorer = _fit   # same hyperparams for both

def _eval(clf, X, y) -> dict:
    p = clf.predict(X)
    return {
        "accuracy":  accuracy_score(y, p),
        "f1":        f1_score(y, p, zero_division=0),
        "precision": precision_score(y, p, zero_division=0),
        "recall":    recall_score(y, p, zero_division=0),
    }

# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------

_DEFAULT_N     = 3
_FALLBACK_HINTS = ["Look for the main detail", "Scan for key terms", "Use the passage context"]


def pick_distractors(passage: str, correct_answer: str, scorer: LogisticRegression, n: int = _DEFAULT_N) -> List[str]:
    pool = _candidate_pool(passage)
    if not pool:
        return []
    m = len(pool)
    mat_p, mat_g, mat_a = _batch_transform(pool, [correct_answer] * m, [passage] * m)
    X = np.array([_dist_feats(pool[i], correct_answer, passage, float(_batch_row_cosine(mat_p, mat_g)[i]),
                               float(_batch_row_cosine(mat_p, mat_a)[i])) for i in range(m)], dtype=np.float32)
    ranked = sorted(zip(scorer.predict_proba(X)[:, 1], pool), reverse=True)
    selected, seen = [], set()
    for _, cand in ranked:
        if cand[:4] not in seen:
            seen.add(cand[:4]); selected.append(cand)
        if len(selected) >= n:
            break
    return selected


def pick_hints(passage: str, question: str, scorer: LogisticRegression, n: int = _DEFAULT_N) -> List[str]:
    sents = sentence_fragments(passage)
    if not sents:
        return _FALLBACK_HINTS[:]
    mat_s, mat_q = _batch_transform(sents, [question] * len(sents))
    sims = _batch_row_cosine(mat_s, mat_q)
    X = np.array([_hint_feats(sents[i], question, i, len(sents), float(sims[i])) for i in range(len(sents))], dtype=np.float32)
    top = [s for _, s in sorted(zip(scorer.predict_proba(X)[:, 1], sents), reverse=True)[:n]]
    hints = []
    if top:       hints.append(f"Hint 1: Think about {' '.join(top_keywords(top[-1], k=4))}")
    if len(top)>1: hints.append(f"Hint 2: {top[1][:120]}")
    if len(top)>2: hints.append(f"Hint 3: {top[0][:150]}")
    return hints

# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------

def run():
    SEP = "=" * 42
    print(f"[scorer_b] starting\n{SEP}")

    # Local paths (commented)
    # train_df = pd.read_csv("../data/processed/train_clean.csv")
    # val_df   = pd.read_csv("../data/processed/val_clean.csv")
    
    # Kaggle paths (active)
    train_df = pd.read_csv("/kaggle/working/data/processed/train_clean.csv")
    val_df   = pd.read_csv("/kaggle/working/data/processed/val_clean.csv")
    print(f"train: {len(train_df):,}  |  val: {len(val_df):,}")

    print("\n--- distractor ---")
    X_d, y_d = _build_distractor_rows(train_df)
    print(f"  matrix {X_d.shape}  positives: {y_d.sum():,}")

    print("\n--- hint ---")
    X_h, y_h = _build_hint_rows(train_df)
    print(f"  matrix {X_h.shape}  positives: {y_h.sum():,}")

    print("\n--- fit ---")
    dist_clf = _fit(X_d, y_d)
    hint_clf = _fit(X_h, y_h)

    # eval with metrics collection
    metrics_rows = []
    for split, build in [("train", lambda: (X_d, y_d, X_h, y_h)),
                          ("val",   lambda: (*_build_distractor_rows(val_df), *_build_hint_rows(val_df)))]:
        Xd, yd, Xh, yh = build()
        for name, clf, X, y in [("distractor", dist_clf, Xd, yd), ("hint", hint_clf, Xh, yh)]:
            m = _eval(clf, X, y)
            metrics_rows.append({
                "model": name,
                "split": split,
                "accuracy": m["accuracy"],
                "f1": m["f1"],
                "precision": m["precision"],
                "recall": m["recall"]
            })
            print(f"  [{name} {split}]  " + "  ".join(f"{k}={v:.4f}" for k, v in m.items()))

    joblib.dump(dist_clf, os.path.join(_SCORER_OUT, "distractor.pkl"))
    joblib.dump(hint_clf, os.path.join(_SCORER_OUT, "hint.pkl"))
    
    # Export metrics to CSV
    _export_metrics_to_csv(metrics_rows)
    
    print(f"\n[scorer_b] done — saved to {_SCORER_OUT}\n{SEP}")


def _export_metrics_to_csv(metrics_rows: list):
    """Export Model-B metrics to CSV."""
    import os
    # Local path (commented)
    # reports_dir = os.path.normpath(os.path.join(_THIS_DIR, "..", "data", "processed", "reports"))
    
    # Kaggle path (active)
    reports_dir = "/kaggle/working/data/processed/reports"
    os.makedirs(reports_dir, exist_ok=True)
    
    df = pd.DataFrame(metrics_rows)
    csv_path = os.path.join(reports_dir, "model_b_metrics.csv")
    df.to_csv(csv_path, index=False)
    print(f"    → {csv_path}")


if __name__ == "__main__":
    run()

[scorer_b] starting
train: 70,270  |  val: 8,784

--- distractor ---


  distractor features: 100%|██████████| 421620/421620 [01:10<00:00, 6010.09ex/s]


  matrix (421620, 6)  positives: 210,810

--- hint ---


  hint features: 100%|██████████| 1165584/1165584 [00:25<00:00, 46261.93ex/s]


  matrix (1165584, 4)  positives: 70,270

--- fit ---
  [distractor train]  accuracy=0.9902  f1=0.9902  precision=0.9940  recall=0.9864
  [hint train]  accuracy=0.8506  f1=0.3947  precision=0.2611  recall=0.8079


  hint features: 100%|██████████| 145031/145031 [00:03<00:00, 46045.89ex/s]


  [distractor val]  accuracy=0.9905  f1=0.9904  precision=0.9938  recall=0.9871
  [hint val]  accuracy=0.8521  f1=0.3984  precision=0.2643  recall=0.8083
    → /kaggle/working/data/processed/reports/model_b_metrics.csv

[scorer_b] done — saved to /kaggle/working/models/model_b/traditional


In [1]:
import shutil
from IPython.display import FileLink

# 1. Create the zip
shutil.make_archive('my_working_files', 'zip', '/kaggle/working')

# 2. Show the link (Ensure there are NO spaces before FileLink)
FileLink(r'my_working_files.zip')

/kaggle/working/my_working_files.zip